In [1]:
# ## Setup
import os

os.environ['CUDA_VISIBLE_DEVICES'] = '0'
# os.environ['CUDA_VISIBLE_DEVICES'] = '1'

In [2]:
import os, sys, gc 
import pandas as pd
import numpy as np
import random
import time
from pathlib import Path 
from sklearn.metrics import log_loss
from sklearn.model_selection import StratifiedKFold, GroupKFold, StratifiedGroupKFold 
from sklearn.preprocessing import OneHotEncoder
import argparse
import collections
import yaml
from types import SimpleNamespace
import re
from sklearn.metrics import accuracy_score

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
from torch.utils.data import SequentialSampler, DataLoader
from torch.cuda.amp import GradScaler, autocast
import torch.utils.checkpoint
from torch.nn import CrossEntropyLoss



import transformers
from transformers import DebertaTokenizer, DebertaForTokenClassification
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import (
    get_cosine_schedule_with_warmup,
    get_linear_schedule_with_warmup,
)

from functools import partial
import multiprocessing as mp
from scipy.special import softmax

from transformers import AutoTokenizer, AutoModel, AutoConfig
from transformers import (AutoModelForTokenClassification,
                          DataCollatorForTokenClassification, 
                          TrainingArguments, 
                          Trainer,
                          TrainerCallback)
from transformers import DebertaModel, DebertaV2Model


import pytorch_lightning as pl
from pytorch_lightning import callbacks
from pytorch_lightning.callbacks.early_stopping import EarlyStopping
from pytorch_lightning.callbacks import ModelCheckpoint


import wandb
import warnings 
warnings.filterwarnings('ignore')
torch.set_warn_always(False)

print('torch.__version__:', torch.__version__)
print('transformers.__version__:', transformers.__version__)
print('PLightning.__version__:', pl.__version__)


os.environ['TOKENIZERS_PARALLELISM'] = 'false'
# os.environ["OMP_NUM_THREADS"] = "1"
# os.environ["MKL_NUM_THREADS"] = "1"
# os.environ["OPENBLAS_NUM_THREADS"] = "1"
# os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
# os.environ["NUMEXPR_NUM_THREADS"] = "1"


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.__version__: 2.2.1
transformers.__version__: 4.30.2
PLightning.__version__: 2.0.9


# Config

In [4]:
class CFG:

    exp_name = 'exp62_V7b' 
    task = "ner"

    n_folds = 5 # 10
    trn_folds = [0] #[0] #[1] #[3] #[2] #[1] #[0] # [0,1,2,3,4]

    # model_name = "microsoft/deberta-base"
    model_name = "microsoft/deberta-large"

    # model_name = "microsoft/deberta-v3-base"
    # model_name = "microsoft/deberta-v3-large"

    # model_name = "allenai/longformer-base-4096"
    # model_name = "allenai/longformer-large-4096"
     
    pretrained_detector_path = '/mnt/md1/Work/dev_IME/Git_hacks/Feedback3/Feedback/checkpoints/tasckj/deberta_large_fold0.pth'
    # pretrained_detector_path = '/mnt/md1/Work/dev_IME/Git_hacks/Feedback3/Feedback/checkpoints/tasckj/deberta_base_fold0.pth'  
    # pretrained_detector_path='/mnt/md1/Work/dev_IME/Git_hacks/Feedback3/Feedback/checkpoints/tasckj'  
    # pretrained_detector_path = 'none'

    pretrained_path = 'none'

    max_length = 1536 #768  
    stride = 128
    min_tokens = 6
    # TRAINING HYPERPARAMS
    batch_size = 1 #32 #4       # 16
    epochs = 5 #4 #1
    gradient_checkpointing = True 
    
    # scheduler = "Linear"
    scheduler = 'cosine'
    LR = 4e-5  #6e-5 #2e-5    # 4e-5    ## 3e-05
    WD = 0.01
    WARMUP = 0.1
    use_custom_trainer = False # set True for custom training loop - TODO
    fp16 = True 
    seed = 42
    train = True
    infer = False
    debug = False # set True for debugging
    wandb = True # False
    wb_project = "NLP-Rater"
    output = "/mnt/md1/Work/dev_IME/Git_hacks/Feedback3/output/"

    accumulate_grad_batches = 4
    GRAD_ACC = 4 #8 #1 #2 #4  #   8 # 4 ## 8
    gradient_clip_val = 1.0

    restart_epoch = 0
    early_stopping = False
    patience = 100 
    hidden_drop_prob = 0.0 
    p_drop = 0.0 
    warmup_ratio = 0.1

    rnn = 'none'
    p_aug = 0.0
    mixup_alpha=1.0
    aug_stop_epoch=999
    adv_sift = False 
    l2norm = False
    awp = False
    awp_lr=1.0 
    awp_eps=0.01
    awp_start_epoch=0
    

    # ######
    # DATA_BASE_DIR = cfg["DATA_BASE_DIR"]
    # LR = cfg["LR"]
    # NUM_EPOCHS = cfg["NUM_EPOCHS"]
    NUM_CORES = 8 #cfg["NUM_CORES"]
    # BATCH_SIZE = cfg["BATCH_SIZE"]
    # USE_FP16 = cfg["USE_FP16"]
    # GRAD_ACCUM_STEPS = cfg["GRAD_ACCUM_STEPS"]
    # MAX_SEQ_LENGTH = cfg["MAX_SEQ_LENGTH"]
    # SPLIT_NUM = cargs.fold
    # PRETRAINED_MODEL = cargs.pretrained_model

    # TRAIN_CSV = os.path.join(DATA_BASE_DIR, 'feedback-prize-2021/train.csv')
    # TRAIN_DIR = os.path.join(DATA_BASE_DIR, 'feedback-prize-2021/train/')

    MIN_TOKENS = {
        "Lead": 6,
        "Position": 3,
        "Evidence": 20,
        "Claim": 1,
        "Concluding Statement": 3,
        "Counterclaim": 7,
        "Rebuttal": 6
    }


In [5]:
os.chdir('/mnt/md1/Work/dev_IME/Git_hacks/Feedback3')
sys.path.append('/mnt/md1/Work/dev_IME/Git_hacks/Feedback3/')
sys.path.append('/mnt/md1/Work/dev_IME/Git_hacks/Feedback3/src/')

from src.utils.utils import *
from src.fp1_sol01.utils import Log, get_f1_score, after_deal
from src.fp1_sol01.utils import get_feat, dataset, Collate


DIR = Path("/mnt/md1/Work/dev_IME/Git_hacks/Feedback3/input/")
OUTPUT = Path(CFG.output + CFG.exp_name)
OUTPUT.mkdir(exist_ok=True, parents=True)

In [6]:
def seed_everything(seed: int):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

In [7]:
# seed everything
CFG.device = device
# set_seed(CFG.seed)
seed_everything(CFG.seed)

In [10]:
discourse_type = [
    'Claim', 'Evidence', 'Position', 'Concluding Statement', 'Lead', 'Counterclaim', 'Rebuttal'
    ]


ner_labels = {
    'O': 0,
    'B-Lead': 1,
    'I-Lead': 2,
    'B-Position': 3,
    'I-Position': 4,
    'B-Evidence': 5,
    'I-Evidence': 6,
    'B-Claim': 7,
    'I-Claim': 8,
    'B-Concluding Statement': 9,
    'I-Concluding Statement': 10,
    'B-Counterclaim': 11,
    'I-Counterclaim': 12,
    'B-Rebuttal': 13,
    'I-Rebuttal': 14
    }

ner_labels_eff = {
    'Non-Effective': 0,
    'Effective': 1,
    }

inverted_ner_labels_eff = dict((v,k) for k,v in ner_labels_eff.items())
inverted_ner_labels_eff[-100] = 'Special Token'


## RATER mapping 
discourse_type_to_int = {
    'Lead': 0,
    'Position': 1,
    'Claim': 2,
    'Evidence': 3, 
    'Counterclaim': 4, 
    'Rebuttal': 5, 
    'Concluding Statement': 6, 
}


CLASSES = list(discourse_type_to_int.keys())

# LABEL2TYPE = ('Lead', 'Position', 'Claim', 'Counterclaim', 'Rebuttal', 'Evidence', 'Concluding Statement')
LABELS_TYPE = [
    'Lead', #0
    'Position', #1 
    'Claim', #2
    'Evidence', #3
    'Counterclaim', #4
    'Rebuttal', #5
    'Concluding Statement' #6 
]

TYPE2LABEL = {t:l for l, t in enumerate(LABELS_TYPE)}
LABEL2TYPE = {k:v for v,k in TYPE2LABEL.items()}
# CLASSES == LABELS_TYPE

# Utils

In [14]:
## FP1-06 YOLO span detector 
## from: https://www.kaggle.com/code/tascj0/a-text-span-detector

from torchvision.ops import roi_align, nms

def to_gpu(data):
    '''
    https://www.kaggle.com/code/tascj0/a-text-span-detector
    '''
    if isinstance(data, dict):
        return {k: to_gpu(v) for k, v in data.items() }
        # return {k: to_gpu(v) for k, v in data.items() if k not in ['text', 'text_id']}
    elif isinstance(data, list):
        return [to_gpu(v) for v in data]
    elif isinstance(data, torch.Tensor):
        return data.cuda()
    else:
        return data

def to_np(t):
    if isinstance(t, torch.Tensor):
        return t.data.cpu().numpy()
    else:
        return t


def aggregate_tokens_to_words(feat, word_boxes):
    feat = feat.permute(0, 2, 1).unsqueeze(2)
    output = roi_align(feat, [word_boxes], 1, aligned=True)
    return output.squeeze(-1).squeeze(-1)


def span_nms(start, end, score, nms_thr=0.5):
    boxes = torch.stack(
        [
            start,
            torch.zeros_like(start),
            end,
            torch.ones_like(start),
        ],
        dim=1,
    ).float()
    keep = nms(boxes, score, nms_thr)
    return keep

In [15]:
### INIT Helpers 
### from mcvv 

def constant_init(module, val, bias=0):
    if hasattr(module, 'weight') and module.weight is not None:
        nn.init.constant_(module.weight, val)
    if hasattr(module, 'bias') and module.bias is not None:
        nn.init.constant_(module.bias, bias)


def xavier_init(module, gain=1, bias=0, distribution='normal'):
    assert distribution in ['uniform', 'normal']
    if distribution == 'uniform':
        nn.init.xavier_uniform_(module.weight, gain=gain)
    else:
        nn.init.xavier_normal_(module.weight, gain=gain)
    if hasattr(module, 'bias') and module.bias is not None:
        nn.init.constant_(module.bias, bias)


def normal_init(module, mean=0, std=1, bias=0):
    nn.init.normal_(module.weight, mean, std)
    if hasattr(module, 'bias') and module.bias is not None:
        nn.init.constant_(module.bias, bias)


def uniform_init(module, a=0, b=1, bias=0):
    nn.init.uniform_(module.weight, a, b)
    if hasattr(module, 'bias') and module.bias is not None:
        nn.init.constant_(module.bias, bias)


def kaiming_init(module,
                 a=0,
                 mode='fan_out',
                 nonlinearity='relu',
                 bias=0,
                 distribution='normal'):
    assert distribution in ['uniform', 'normal']
    if distribution == 'uniform':
        nn.init.kaiming_uniform_(
            module.weight, a=a, mode=mode, nonlinearity=nonlinearity)
    else:
        nn.init.kaiming_normal_(
            module.weight, a=a, mode=mode, nonlinearity=nonlinearity)
    if hasattr(module, 'bias') and module.bias is not None:
        nn.init.constant_(module.bias, bias)


def caffe2_xavier_init(module, bias=0):
    # `XavierFill` in Caffe2 corresponds to `kaiming_uniform_` in PyTorch
    # Acknowledgment to FAIR's internal code
    kaiming_init(
        module,
        a=1,
        mode='fan_in',
        nonlinearity='leaky_relu',
        distribution='uniform')


def bias_init_with_prob(prior_prob):
    """initialize conv/fc bias value according to giving probablity."""
    bias_init = float(-np.log((1 - prior_prob) / prior_prob))
    return bias_init

In [16]:
def get_hf_scheduler(cfg, optimizer, total_steps):
    if cfg.scheduler == "Linear":
        scheduler = get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps=int(
                cfg.WARMUP * (total_steps // cfg.batch_size)
            ),
            num_training_steps=cfg.epochs
            * (total_steps // cfg.batch_size),
        )
    else:
        scheduler = get_cosine_schedule_with_warmup(
            optimizer,
            num_warmup_steps=int(
                cfg.WARMUP * (total_steps // cfg.batch_size)
            ),
            num_training_steps=cfg.epochs
            * (total_steps // cfg.batch_size),
        )
    return scheduler



def get_cos_scheduler(args, optimizer):
    scheduler = CosineAnnealingWarmupRestarts(
        optimizer = optimizer,
        first_cycle_steps = args.num_train_steps, 
        cycle_mult = 1,
        max_lr = args.max_lr, 
        min_lr = args.min_lr, 
        warmup_steps = args.num_train_steps * 0.2,
        gamma = 1.,
        last_epoch = -1
    )
    return scheduler

In [17]:
def get_data(fold=0):
    
    df = pd.read_csv(DIR / "train_sgkf_fold5.csv")
    df['essay_id'] = df['id'] = df['essay_id_comp']
    df['discourse_effectiveness_int'] = df['discourse_effectiveness'].map(ner_labels_eff)

    ## add whole essays 
    df_texts = pd.read_csv('/mnt/md1/Work/dev_IME/Git_hacks/Feedback3/input/train_fold10_sgkf.csv')
    df_texts = df_texts[['essay_id', 'discourse_id', 'essay_text']]
    df = df.merge(df_texts, on=['essay_id', 'discourse_id'])

    if "class" not in df.columns:
        df["class"] = df["discourse_type"]

    print(df.shape)
    display(df.head())

    train_df = df[df["fold"] != fold].reset_index(drop=True)
    valid_df = df[df["fold"] == fold].reset_index(drop=True)

    # # if len(test_df) == 0:
    # #     sample_id = train_df['id'].drop_duplicates().sample(frac=0.2).values
    # #     test_df = train_df[train_df['id'].isin(sample_id)].reset_index(drop=True)
    # log('fold:', fold)
    # log('train.shape:', train_df.shape,'\t','test.shape:', valid_df.shape)
    # log('train ids:', train_df.id.nunique(),'\t','test ids:', valid_df.id.nunique())

    return train_df, valid_df
        

# Data

In [ ]:
train_df, valid_df = get_data(fold=CFG.trn_folds[0])

print('train.shape:', train_df.shape,'\t','test.shape:', valid_df.shape)

In [ ]:
valid_df['fold'].unique()

In [ ]:
from transformers import AutoModel, AutoTokenizer
from transformers.models.deberta_v2 import DebertaV2TokenizerFast

# tokenizer = AutoTokenizer.from_pretrained(CFG.model_name)
# if 'deberta-v' in CFG.model_name or 'bigbird' in CFG.model_name:
#     tokenizer.add_tokens('\n')


if 'deberta-v2' in CFG.model_name or 'deberta-v3' in CFG.model_name:    
    tokenizer = DebertaV2TokenizerFast.from_pretrained(CFG.model_name, trim_offsets=False)
    special_tokens_dict = {'additional_special_tokens': ['\n\n'] + [f'[{s.upper()}]' for s in CLASSES]}
    _ = tokenizer.add_special_tokens(special_tokens_dict)
else:
    tokenizer = AutoTokenizer.from_pretrained(CFG.model_name, trim_offsets=False)
    special_tokens_dict = {'additional_special_tokens': [f'[{s.upper()}]' for s in CLASSES]} # list(relation_mapper.keys()) 
    _ = tokenizer.add_special_tokens(special_tokens_dict)

In [ ]:
special_tokens_dict

In [ ]:
CFG.tokenizer = tokenizer

In [ ]:
tokenizer.save_pretrained(OUTPUT / "tokenizer")

In [ ]:
tokenizer

# Dataset 

In [ ]:
from src.fp1_sol01.utils import get_offset_mapping, is_head


def encode2(text, tokenizer, data, labels_to_ids, labels_to_ids2):
    if 'deberta-v' in tokenizer.name_or_path:
        encoding = tokenizer.encode_plus(text)
        tokens = tokenizer.convert_ids_to_tokens(encoding['input_ids'])
        encoding['offset_mapping'] = get_offset_mapping(text, tokens)
    else:
        encoding = tokenizer.encode_plus(text, return_offsets_mapping=True)
    input_ids      = encoding['input_ids']
    attention_mask = encoding['attention_mask']
    offset_mapping = encoding['offset_mapping']
    
    unmask_length   = sum(attention_mask)
    token_label     = [0] * unmask_length

    token_label_eff = [0] * unmask_length

    if 'discourse_start' in data.columns:
        data = data.sort_values('discourse_start').reset_index(drop=True)
        
        char_label = list(
            zip(data.discourse_start.astype(int), data.discourse_end.astype(int), data.discourse_type, data.discourse_effectiveness)
            )
        
        # new_char_label = []
        # last_idx = 0
        # for c_label in char_label:
        #     if c_label[0] != last_idx:
        #         if len(text[last_idx:c_label[0]].split())>0:
        #             new_char_label.append([last_idx, c_label[0], 'O'])
        #     new_char_label.append(c_label)
        #     last_idx = c_label[1]
        # if c_label[1] != len(text):
        #     if len(text[c_label[1]:len(text)].split())>0:
        #         new_char_label.append([c_label[1], len(text), 'O'])
        # char_label = new_char_label

        is_first = True
        for i, mapping in enumerate(offset_mapping):
            while len(char_label) > 0 and mapping[1] > char_label[0][1]:
                char_label.pop(0)                        # 大于区间
                is_first = True
            if len(char_label) == 0:                     # char_label 为空，暂停
                break
            if mapping[1] >= char_label[0][0] and mapping != (0,0):           # 介于区间
                if is_first:
                    if is_head(text[mapping[0]:mapping[1]]):  # 是否合法的开头：剔除空值，和上句的标点符合
                        token_label[i]     = labels_to_ids['B-'+char_label[0][2]]
                        token_label_eff[i] = labels_to_ids2[char_label[0][3]]
                        is_first = False
                else:
                    token_label[i]     = labels_to_ids['I-'+char_label[0][2]]  # 后续同样token
                    token_label_eff[i] = labels_to_ids2[char_label[0][3]]
                                                         # 小于区间不操作
    return input_ids,attention_mask,token_label,token_label_eff,offset_mapping
                              

### ++
def get_feat_helper2(args, tokenizer, df, train_ids, text_col='full_text'):
    
    grp_text = df.groupby('id')[text_col].apply(lambda x: x.values[0]).reset_index()
    
    training_samples = []
    for idx in tqdm(train_ids):
        # text = df[df['id']==idx][text_col].values[0].rstrip()
        text = grp_text[grp_text['id']==idx][text_col].values[0].rstrip()
        input_ids, attention_mask, token_label, token_label_eff, offset_mapping = encode2(
            text, tokenizer, df[df['id']==idx], 
            args.labels_to_ids,
            args.labels_to_ids2
            )
        training_samples.append({
            'id': idx,
            'text': text,
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'token_label': token_label,
            'token_label_eff': token_label_eff,
            'offset_mapping': offset_mapping
        })
    return training_samples


def get_feat(df, tokenizer, args, data_key, text_col='full_text'):
    num_jobs = 8
    data = []
    train_ids = df["id"].unique()

    train_ids_splits = np.array_split(train_ids, num_jobs)
    results = Parallel(n_jobs=num_jobs, backend="multiprocessing")(
        delayed(get_feat_helper2)(args, tokenizer, df, idx, text_col) for idx in train_ids_splits
    )
    for result in results:
        data.extend(result)

    data = pd.DataFrame(sorted(data, key=lambda x: len(x['input_ids'])))
    return data

In [ ]:
class CustomCollator(object):
    def __init__(self, tokenizer, model_name): #model
        self.pad_token_id = tokenizer.pad_token_id

        config_model = config = AutoConfig.from_pretrained(model_name)
        # if hasattr(model.config, 'attention_window'):
        if hasattr(config_model, 'attention_window'):
            # For longformer
            # https://github.com/huggingface/transformers/blob/v4.17.0/src/transformers/models/longformer/modeling_longformer.py#L1548
            self.attention_window = (
                config_model.attention_window if isinstance(
                    config_model.attention_window, int)
                else max(config_model.attention_window)
                )
        else:
            self.attention_window = None

    def __call__(self, samples):
        batch_size = len(samples)
        assert batch_size == 1, f'Only batch_size=1 supported, got batch_size={batch_size}.'

        sample = samples[0]

        max_seq_length = len(sample['input_ids'])
        if self.attention_window is not None:
            attention_window = self.attention_window
            padded_length = (attention_window -
                             max_seq_length % attention_window
                             ) % attention_window + max_seq_length
        else:
            padded_length = max_seq_length

        input_shape = (1, padded_length)
        input_ids = torch.full(input_shape,
                               self.pad_token_id,
                               dtype=torch.long)
        attention_mask = torch.zeros(input_shape, dtype=torch.long)
        # labels_eff = torch.zeros(input_shape, dtype=torch.long)

        seq_length = len(sample['input_ids'])
        input_ids[0, :seq_length] = sample['input_ids']
        attention_mask[0, :seq_length] = sample['attention_mask']
        # labels_eff[0, :seq_length] = sample['labels_eff'][:seq_length]

        text_id = sample['text_id']
        text = sample['text']
        word_boxes = sample['word_boxes']
        gt_spans = sample['gt_spans']
        
        ## ++
        span_list = [sample['span_list']]
        discourse_types = [sample['discourse_types']]
        # labels_effectiveness = sample['labels_eff']

        # o = {}
        # for k in sample.keys():
        #     if k in ['span_list', 'discourse_types']:
        #         o[k] = [sample[k] for sample in samples]


        return dict(
            text_id=text_id,
            text=text,
            input_ids=input_ids,
            attention_mask=attention_mask,
            word_boxes=word_boxes,
            gt_spans=gt_spans,
            ## ++
            span_list=span_list,
            discourse_types=discourse_types,
            # labels_effectiveness=labels_effectiveness
            # labels_eff=labels_eff
            )
    

In [ ]:
# accepts file path, returns tuple of (file_ID, txt, NER labels)
def generate_NER_labels_for_file(uuid, df, id_col='essay_id'):

    curr_id = uuid 
    curr_txt = df[df[id_col]==uuid]['essay_text'].values[0].rstrip()

    # Set all token labels initially to non-label
    curr_labels = [ner_labels['O']] * len(curr_txt)

    # curr_labels_eff = [ner_labels_eff['O']] * len(curr_txt)
    curr_labels_eff = [-100] * len(curr_txt)
    
    # Iterate through all labels associated w/ file and update labels
    curr_df = df[df[id_col]==curr_id]
    for curr_discourse in curr_df.itertuples():

        curr_discourse_label     = curr_discourse.discourse_type 
        curr_discourse_label_eff = curr_discourse.discourse_effectiveness 

        for curr_txt_idx in range(curr_discourse.discourse_start, 
                                  min(curr_discourse.discourse_end+1, len(curr_labels))):
            if curr_txt_idx == curr_discourse.discourse_start:
                iob_label     = ner_labels['B-' + curr_discourse_label]
                # iob_label_eff = ner_labels_eff['B-' + curr_discourse_label_eff]

            else:
                iob_label     = ner_labels['I-' + curr_discourse_label]
                # iob_label_eff = ner_labels_eff['I-' + curr_discourse_label_eff]

            iob_label_eff = ner_labels_eff[curr_discourse_label_eff]

            curr_labels[curr_txt_idx]     = iob_label
            curr_labels_eff[curr_txt_idx] = iob_label_eff

    assert curr_labels != [ner_labels['O']] * len(curr_txt)
    # assert curr_labels_eff != [ner_labels_eff['O']] * len(curr_txt)
    assert curr_labels_eff != [-100] * len(curr_txt)

    # return curr_id, curr_txt, curr_labels, curr_labels_eff
    return curr_labels_eff

In [ ]:
# [tokenizer.convert_tokens_to_ids(f'[{discourse_type.upper()}]') for discourse_type in CLASSES]

In [ ]:
# tokenizer.convert_tokens_to_ids(f'POSITION')
# tokenizer.convert_tokens_to_ids(f'NEW TYPE')

In [ ]:
# tokenizer.vocab  
# inv_vocab = {k: v for v,k in tokenizer.vocab.items()}
# inv_vocab[3] #'[UNK]'

In [ ]:
### FP2 2nd sol dataset 
# https://github.com/Shujun-He/TeamSKT-Feedback-Prize---Predicting-Effective-Arguments-2nd-Place-solution/blob/main/Shujun_solution/src/Feedback_Prize/Dataset.py

class Feedback2Dataset_02(Dataset):
    def __init__(self, df, tokenizer, mask_prob=0, mask_ratio=0, aug='false'):
        self.df = df
        self.unique_ids = sorted(df['essay_id'].unique())
        self.tokenizer = tokenizer
        self.mask_prob = mask_prob
        self.mask_ratio = mask_ratio
        self.aug = aug
        self.discourse_type_token_ids_dict = {
            discourse_type : tokenizer.convert_tokens_to_ids(f'[{discourse_type.upper()}]')
            for discourse_type in LABEL2TYPE
        }

        self.inv_discourse_type_token_ids_dict = {v:k for k,v in self.discourse_type_token_ids_dict.items()}
        
    def __len__(self):
        return len(self.unique_ids)
    
    def __getitem__(self, idx):
        essay_id = self.unique_ids[idx]
        sample_df = self.df[self.df['essay_id']==essay_id].reset_index(drop=True)
        discourse_ids = sample_df['discourse_id'].values

        text = ''
        discourse_types = []
        labels = []
        for discourse_type, discourse_text, label in zip(
            sample_df['discourse_type'].values, sample_df['discourse_text'].values, sample_df['discourse_effectiveness_int'].values
        ):
            text += f' [{discourse_type.upper()}] {discourse_text}'
            discourse_types.append(discourse_type.upper())
            labels.append(label)

        tokens = self.tokenizer.encode_plus(text, add_special_tokens=True)
        input_ids = torch.LongTensor(tokens['input_ids'])
        attention_mask = torch.ones(len(input_ids)).long()
        
        span_list = []
        for i_token, input_id in enumerate(input_ids):
            if input_id in self.discourse_type_token_ids_dict.values():
                span_list.append(i_token)
        
        # random masking
        if np.random.random() < self.mask_prob:
            all_inds = np.arange(1, len(input_ids)-1)
            n_mask = max(int(len(all_inds) * self.mask_ratio), 1)
            np.random.shuffle(all_inds)
            mask_inds = np.array([inds for inds in all_inds[:n_mask] if inds not in span_list])
            input_ids[mask_inds] = self.tokenizer.mask_token_id
        
        return dict(
            essay_id = essay_id,
            discourse_ids = discourse_ids,
            text = text,
            input_ids = input_ids,
            attention_mask = attention_mask,
            discourse_types = discourse_types,
            label = labels,
            span_list = span_list,
        )
    



class Feedback2Collator_02(object):
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
        
    def __call__(self, samples):
        output = dict()
        
        for k in samples[0].keys():
            output[k] = [sample[k] for sample in samples]
        
        # calculate max token length of this batch
        batch_max = max([len(ids) for ids in output["input_ids"]])
        
        # add padding
        if self.tokenizer.padding_side == "right":
            output["input_ids"] = [s.tolist() + (batch_max - len(s)) * [self.tokenizer.pad_token_id] for s in output["input_ids"]]
            output["attention_mask"] = [s.tolist() + (batch_max - len(s)) * [0] for s in output["attention_mask"]]
        else:
            output["input_ids"] = [(batch_max - len(s)) * [self.tokenizer.pad_token_id] + s.tolist() for s in output["input_ids"]]
            output["attention_mask"] = [(batch_max - len(s)) * [0] + s.tolist() for s in output["attention_mask"]]
            
        output["input_ids"] = torch.LongTensor(output["input_ids"])
        output["attention_mask"] = torch.LongTensor(output["attention_mask"])
        
        if "label" in output.keys():
            output["label"] = torch.LongTensor(output["label"])
        if "label2" in output.keys():
            output["label2"] = torch.LongTensor(output["label2"])
        return output

In [ ]:
if CFG.debug:

    ds = Feedback2Dataset_02(train_df, CFG.tokenizer)
    dl = DataLoader(ds, batch_size=1, shuffle=True, collate_fn=Feedback2Collator_02(CFG.tokenizer))

# ds[0].keys()
##dict_keys(['essay_id', 'discourse_ids', 'text', 'input_ids', 'attention_mask', 'discourse_types', 'label', 'span_list'])

In [ ]:
if CFG.debug:
    s0 = ds[20]
    for k,v in s0.items():
        if k not in ['text', 'text_id', 'discourse_ids', 'essay_id', 'discourse_types', 'label', 'span_list']:
            print(k, v.shape)
        else:
            print(k, v)

        print(ds.discourse_type_token_ids_dict)

In [ ]:
if CFG.debug:
    s1 = next(iter(dl))
    for k,v in s1.items():
        if k not in ['text', 'text_id', 'discourse_ids', 'essay_id', 'discourse_types', 'label', 'span_list']:
            print(k, v.shape)
        else:
            print(k, v)

In [ ]:
# ds.discourse_type_token_ids_dict

In [ ]:
# for i_token, input_id in enumerate(s0['input_ids']):
#     if input_id in ds.discourse_type_token_ids_dict.values():
#         print(i_token, input_id)

In [ ]:
# for i_token, input_id in enumerate(s['input_ids']):
#     if input_id in ds.discourse_type_token_ids_dict.values():
#         print(i_token, input_id)

In [ ]:
class SpanDetectorDataset(Dataset):
    def __init__(self, df,
                 tokenizer,
                 mask_prob=0.0,
                 mask_ratio=0.0):
        
        self.df = df.copy() #pd.read_csv(csv_file)
        self.samples = list(self.df.groupby('id'))
        self.tokenizer = tokenizer
        print(f'Loaded {len(self)} samples.')

        assert 0 <= mask_prob <= 1
        assert 0 <= mask_ratio <= 1
        self.mask_prob = mask_prob
        self.mask_ratio = mask_ratio
        ###
        self.discourse_type_token_ids_dict = {
            discourse_type : tokenizer.convert_tokens_to_ids(f'[{discourse_type.upper()}]')
            for discourse_type in CLASSES
        }
        self.inv_discourse_type_token_ids_dict = {v:k for k,v in self.discourse_type_token_ids_dict.items()}



    def __len__(self):
        return len(self.samples)



    def __getitem__(self, index):
        text_id, df = self.samples[index]
        
        # text_path = os.path.join(self.text_dir, f'{text_id}.txt')
        # with open(text_path) as f:
        #     text = f.read().rstrip()

        text = df['essay_text'].values[0].rstrip()

        discourse_types = [] ## TODO replace 
        ############## Tom's sol 
        # if 'discourse_type' in df.columns:
        #     text = ''
        #     discourse_types = []
        #     labels_eff = []
        #     for disc_type_, disc_text_, lbl_eff_ in zip(
        #         df['discourse_type'].values, df['discourse_text'].values, df['discourse_effectiveness_int'].values
        #     ):
        #         text += f' [{disc_type_.upper()}] {disc_text_}'
        #         discourse_types.append(disc_type_.upper())
        #         labels_eff.append(lbl_eff_)

        # else:
        #     text = df['essay_text'].values[0].rstrip()
        #     #  * len() 

        # tokens = self.tokenizer.encode_plus(text, return_offsets_mapping=True, add_special_tokens=True)
        ##################################
        
        tokens = self.tokenizer(text, return_offsets_mapping=True)
        input_ids = torch.LongTensor(tokens['input_ids'])
        attention_mask = torch.ones(len(input_ids)).long()
        
        # attention_mask = torch.LongTensor(tokens['attention_mask'])
        offset_mapping = np.array(tokens['offset_mapping'])
        offset_mapping = self.strip_offset_mapping(text, offset_mapping)
        num_tokens = len(input_ids)


        # token slices of words
        woff = self.get_word_offsets(text)
        toff = offset_mapping
        wx1, wx2 = woff.T
        tx1, tx2 = toff.T
        ix1 = np.maximum(wx1[..., None], tx1[None, ...])
        ix2 = np.minimum(wx2[..., None], tx2[None, ...])
        ux1 = np.minimum(wx1[..., None], tx1[None, ...])
        ux2 = np.maximum(wx2[..., None], tx2[None, ...])
        ious = (ix2 - ix1).clip(min=0) / (ux2 - ux1 + 1e-12)
        assert (ious > 0).any(-1).all()

        word_boxes = []
        for row in ious:
            inds = row.nonzero()[0]
            word_boxes.append([inds[0], 0, inds[-1] + 1, 1])
        word_boxes = torch.FloatTensor(word_boxes)

        # word slices of ground truth spans
        gt_spans = []
        for _, row in df.iterrows():
            winds = row['predictionstring'].split()
            start = int(winds[0])
            end = int(winds[-1])
            span_label = TYPE2LABEL[row['discourse_type']]
            # gt_spans.append([start, end + 1, span_label])
            eff_label = ner_labels_eff[row['discourse_effectiveness']]
            gt_spans.append([start, end + 1, span_label, eff_label])

        gt_spans = torch.LongTensor(gt_spans)

        # random mask augmentation
        if np.random.random() < self.mask_prob:
            all_inds = np.arange(1, len(input_ids) - 1)
            n_mask = max(int(len(all_inds) * self.mask_ratio), 1)
            np.random.shuffle(all_inds)
            mask_inds = all_inds[:n_mask]
            input_ids[mask_inds] = self.tokenizer.mask_token_id



        #### TODO replace with gt_spans[:,0] --> change to orginal 
        # span_list = []
        # for i_token, input_id in enumerate(input_ids):
        #     if input_id in self.discourse_type_token_ids_dict.values():
        #         span_list.append(i_token)

        span_list = np.array(gt_spans)[:,0].tolist()
        
        if len(discourse_types)==0:
            discourse_types = np.array(gt_spans)[:,2].tolist()
            
        # ### Effective NER labels (3 class)
        # labels_eff = generate_NER_labels_for_file(text_id, self.df) # df
        # labels_eff = torch.LongTensor(labels_eff)

        return dict(
            text=text,
            text_id=text_id,
            input_ids=input_ids,
            attention_mask=attention_mask,
            word_boxes=word_boxes,
            gt_spans=gt_spans,
            ### ++ Tom 
            span_list=span_list,
            discourse_types=discourse_types,
            # labels_eff=labels_eff
                    )

    def strip_offset_mapping(self, text, offset_mapping):
        ret = []
        for start, end in offset_mapping:
            match = list(re.finditer('\S+', text[start:end]))
            if len(match) == 0:
                ret.append((start, end))
            else:
                span_start, span_end = match[0].span()
                ret.append((start + span_start, start + span_end))
        return np.array(ret)

    def get_word_offsets(self, text):
        matches = re.finditer("\S+", text)
        spans = []
        words = []
        for match in matches:
            span = match.span()
            word = match.group()
            spans.append(span)
            words.append(word)
        assert tuple(words) == tuple(text.split())
        return np.array(spans)


In [ ]:
train_ds = SpanDetectorDataset(train_df, tokenizer=CFG.tokenizer, mask_prob=0.5, mask_ratio=0.3)
valid_ds = SpanDetectorDataset(valid_df, tokenizer=CFG.tokenizer)
collate = CustomCollator(CFG.tokenizer, CFG.model_name) #detector.model

train_loader = DataLoader(train_ds,
                      batch_size=1,
                      shuffle=True,
                      num_workers=2,
                      collate_fn=collate)

valid_loader = DataLoader(valid_ds,
                      batch_size=1,
                      shuffle=False,
                      num_workers=2,
                      collate_fn=collate)

In [ ]:
train_ds[0].keys()

In [ ]:
s = next(iter(train_loader))
# s = train_ds[20]

for k,v in s.items():
    if k not in ['text', 'text_id', 'span_list', 'discourse_types']:
        print(k, v.shape)
    else:
        print(k, v)

In [ ]:
# CFG.batch_size = 1 #args.train_batch_size
CFG.num_train_steps = int(len(train_ds) * CFG.epochs / CFG.batch_size)

In [ ]:
CFG.batch_size

In [ ]:
CFG.num_train_steps

# Metric

In [ ]:
### RATER metric 

def remove_overlaps(segments):
    """
    Removes overlaps in predicted segments.

    Args:
    segments (list of tuples): Each tuple represents a segment and contains two elements - the start and end of the segment.

    Returns:
    list of tuples: The processed segments with overlaps removed.
    """
    # Sort segments based on their starting position
    segments.sort(key=lambda x: x[0])

    used_words = set()
    processed_segments = []

    for segment in segments:
        start, end = segment

        # Check for overlap with used words
        overlap = set(range(start, end+1)) & used_words

        # If there's an overlap, remove the used words from the current segment
        if overlap:
            new_segment = set(range(start, end+1)) - overlap

            # If the resulting segment is valid, add it to the list of processed segments
            if len(new_segment) > 1:
                processed_segments.append((min(new_segment), max(new_segment)))

            # Add the new words to the set of used words
            used_words.update(new_segment)
        else:
            # If there's no overlap, add the segment as is to the list of processed segments
            processed_segments.append(segment)

            # Add the words in the current segment to the set of used words
            used_words.update(range(start, end+1))

    return processed_segments

In [ ]:
def calc_overlap(row):
    set_pred = set(row.predictionstring_pred.split(' '))
    set_gt = set(row.predictionstring_gt.split(' '))
    len_gt = len(set_gt)
    len_pred = len(set_pred)
    inter = len(set_gt.intersection(set_pred))
    return inter/max(len_gt,len_pred)

from sklearn.metrics import log_loss, roc_auc_score


def get_f1_score_v1(pred_df, test_df, log=print, slient=True):
    
    test_pred = pred_df.copy()
    
    if not slient:
        log('test_pred.shape:',test_pred.shape,'\ttest_df.shape:',test_df.shape,)
        log('pred class:\n',test_pred['class'].value_counts())
        log('true class:\n',test_df['discourse_type'].value_counts())

    # if 'class' not in test_pred.columns:
    #     test_pred['class'] = test_pred['discourse_type']
    # test_pred['class'] = test_pred['discourse_type']

    f1s = []
    for c in sorted(test_pred['class'].unique()):
        pred_df = test_pred.loc[test_pred['class']==c].copy()
        gt_df = test_df.loc[test_df['discourse_type']==c].copy()

        gt_df = gt_df[['id','discourse_type','predictionstring']].reset_index(drop=True).copy()
        pred_df = pred_df[['id','class','predictionstring']].reset_index(drop=True).copy()
        pred_df['pred_id'] = pred_df.index
        gt_df['gt_id'] = gt_df.index
        # Step 1. all ground truths and predictions for a given class are compared.
        joined = pred_df.merge(gt_df,
                               left_on=['id','class'],
                               right_on=['id','discourse_type'],
                               how='outer',
                               suffixes=('_pred','_gt')
                              )
        joined['predictionstring_gt'] = joined['predictionstring_gt'].fillna(' ')
        joined['predictionstring_pred'] = joined['predictionstring_pred'].fillna(' ')

        # print(joined.head())
        joined['min_overlaps'] = joined.apply(calc_overlap, axis=1)
        joined['potential_TP'] = (joined['min_overlaps'] >= 0.5)

        matched_pred_ids = joined.query('potential_TP')['pred_id'].unique()
        fp_pred_ids = [p for p in joined['pred_id'].unique() if p not in matched_pred_ids]

        matched_gt_ids = joined.query('potential_TP')['gt_id'].unique()
        fn_gt_ids = [c for c in joined['gt_id'].unique() if c not in matched_gt_ids]

        # Get numbers of each type
        TP = len(matched_gt_ids)
        FP = len(fp_pred_ids)
        FN = len(fn_gt_ids)
        #calc microf1
        f1_score = TP / (TP + 0.5*(FP+FN))
        
        if not slient:
            print(f'{c:<20} f1 score:\t{f1_score}')
        f1s.append(f1_score)
    
    print('\nOverall f1 score \t',  np.mean(f1s))

    ### Calculate the precision, recall, and F1 score
    precision = TP / (TP + FP) if TP + FP > 0 else 0
    recall = TP / (TP + FN) if TP + FN > 0 else 0
    # F1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0

    if not slient:
        print(f"TP: {TP}, FP: {FP}, FN: {FN}")
        print(f"pre: {precision}, rec: {recall}")

    return np.mean(f1s)


In [ ]:
def get_f1_score(pred_df, test_df, log=print, slient=True):
    
    test_pred = pred_df.copy()
    
    if not slient:
        log('test_pred.shape:',test_pred.shape,'\ttest_df.shape:',test_df.shape,)
        log('pred class:\n',test_pred['class'].value_counts())
        log('true class:\n',test_df['discourse_type'].value_counts())
    

    test_pred['class'] = test_pred['discourse_type']

    f1s = []
    aucs = []
    logloss_scores = []
    total_scores = []
    for c in sorted(test_pred['class'].unique()):
        pred_df = test_pred.loc[test_pred['class']==c].copy()
        gt_df = test_df.loc[test_df['discourse_type']==c].copy()

        gt_df = gt_df[['id','discourse_type','predictionstring','discourse_effectiveness_int']].reset_index(drop=True).copy()
        pred_df = pred_df[['id','class','predictionstring', 'score_eff']].reset_index(drop=True).copy()
        
        pred_df['pred_id'] = pred_df.index
        gt_df['gt_id'] = gt_df.index
        
        # Step 1. all ground truths and predictions for a given class are compared.
        joined = pred_df.merge(gt_df,
                               left_on=['id','class'],
                               right_on=['id','discourse_type'],
                               how='outer',
                               suffixes=('_pred','_gt')
                              )
        joined['predictionstring_gt'] = joined['predictionstring_gt'].fillna(' ')
        joined['predictionstring_pred'] = joined['predictionstring_pred'].fillna(' ')

        # print(joined.head())
        joined['min_overlaps'] = joined.apply(calc_overlap, axis=1)
        joined['potential_TP'] = (joined['min_overlaps'] >= 0.5)

        matched_pred_ids = joined.query('potential_TP')['pred_id'].unique()
        fp_pred_ids = [p for p in joined['pred_id'].unique() if p not in matched_pred_ids]

        matched_gt_ids = joined.query('potential_TP')['gt_id'].unique()
        fn_gt_ids = [c for c in joined['gt_id'].unique() if c not in matched_gt_ids]

        # Get numbers of each type
        TP = len(matched_gt_ids)
        FP = len(fp_pred_ids)
        FN = len(fn_gt_ids)
        #calc microf1
        f1_score = TP / (TP + 0.5*(FP+FN))
        

        ###########
        ll_score = log_loss(
            joined[joined.potential_TP==True]['discourse_effectiveness_int'],
            joined[joined.potential_TP==True]['score_eff'],
            labels=[0,1]
        )
        logloss_scores.append(ll_score)

        auc_sc = roc_auc_score(
            joined[joined.potential_TP==True]['discourse_effectiveness_int'],
            joined[joined.potential_TP==True]['score_eff']
        )
        aucs.append(auc_sc)

        # #### try this
        # joined3 = joined[['potential_TP', 'discourse_effectiveness_int', 'score_eff']].dropna()
        # auc_sc2 = roc_auc_score(joined3['discourse_effectiveness_int'], joined3['score_eff'],)

        tot_score = 0.5*f1_score + 0.5*ll_score
        total_scores.append(tot_score)

        if not slient:
            print(f'{c:<20} f1 score:\t{f1_score}')
            print(f'{c:<20} logloss (TP) score:{ll_score:.4f} \t auc (TP) score: {auc_sc:.4f}')
            print(f'{c:<20} total score:\t{tot_score}')
            print()
        f1s.append(f1_score)

        ### Calculate the precision, recall, and F1 score
        precision = TP / (TP + FP) if TP + FP > 0 else 0
        recall = TP / (TP + FN) if TP + FN > 0 else 0
        # F1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0

        if not slient:
            print(f"TP: {TP}, FP: {FP}, FN: {FN}")
            print(f"pre: {precision}, rec: {recall}")
    
    print('\nOverall f1 score',  np.mean(f1s))
    print('Overall total score:\t', np.mean(total_scores))
    print()
    print('Overall logloss (TP) score:\t', np.mean(logloss_scores))
    print('Overall auc (TP) score:\t', np.mean(aucs))
    return np.mean(f1s)

# Train Engine

## Stage 1

In [18]:
import time
import pandas as pd
import numpy as np
import gc
from os.path import join as opj
from tqdm import tqdm
import re
import torch
from torch import nn, optim
from torch.utils.data import DataLoader
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.ops import roi_align, nms
from transformers import AutoTokenizer
from transformers import (AutoModelForTokenClassification, AutoModel, AutoTokenizer,
                          AutoConfig, AdamW, get_cosine_schedule_with_warmup, get_cosine_with_hard_restarts_schedule_with_warmup,
                          get_linear_schedule_with_warmup)
# from public_metric_2021 import score_feedback_comp
from collections import OrderedDict

from src.losses.jaccard import *
from src.utils.RATER_MetricCode import compute_scores 

# LABEL2TYPE = ('Lead', 'Position', 'Claim', 'Counterclaim', 'Rebuttal',  'Evidence', 'Concluding Statement')
# TYPE2LABEL = {t:l for l, t in enumerate(LABEL2TYPE)}
# LABEL2TYPE = {l:t for t,l in TYPE2LABEL.items()}


def to_gpu(data):
    if isinstance(data, dict):
        return {k: to_gpu(v) for k, v in data.items()}
    elif isinstance(data, list):
        return [to_gpu(v) for v in data]
    elif isinstance(data, torch.Tensor):
        return data.cuda()
    else:
        return data


def to_np(t):
    if isinstance(t, torch.Tensor):
        return t.data.cpu().numpy()
    else:
        return t


def aggregate_tokens_to_words(feat, word_boxes):
    feat = feat.permute(0, 2, 1).unsqueeze(2)
    output = roi_align(feat, [word_boxes], 1, aligned=True)
    return output.squeeze(-1).squeeze(-1)


def aggregate_tokens_to_words(feat, word_boxes):
    feat = feat.permute(0, 2, 1).unsqueeze(2)
    output = roi_align(feat, [word_boxes], 1, aligned=True)
    return output.squeeze(-1).squeeze(-1)



def span_nms(start, end, score, nms_thr=0.5):
    boxes = torch.stack(
        [
            start,
            torch.zeros_like(start),
            end,
            torch.ones_like(start),
        ],
        dim=1,
    ).float()
    keep = nms(boxes, score, nms_thr)
    return keep



In [19]:
class DatasetTrain(Dataset):
    def __init__(self,
                 df,
                #text_dir,
                 tokenizer,
                 mask_prob=0.0,
                 mask_ratio=0.0,
                ):
        
        self.df = df
        self.samples = list(self.df.groupby('id'))
        # self.text_dir = text_dir
        self.tokenizer = tokenizer
        print(f'Loaded {len(self)} samples.')

        assert 0 <= mask_prob <= 1
        assert 0 <= mask_ratio <= 1
        self.mask_prob = mask_prob
        self.mask_ratio = mask_ratio

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        text_id, df = self.samples[index]
        
        # text_path = opj(self.text_dir, f'{text_id}.txt')
        # with open(text_path) as f:
        #     text = f.read().rstrip()

        text = df['essay_text'].values[0].rstrip()

        tokens = self.tokenizer(text, return_offsets_mapping=True)
        input_ids = torch.LongTensor(tokens['input_ids'])
        attention_mask = torch.LongTensor(tokens['attention_mask'])
        offset_mapping = np.array(tokens['offset_mapping'])
        offset_mapping = self.strip_offset_mapping(text, offset_mapping)
        num_tokens = len(input_ids)

        # token slices of words (offset_mapping = character slices of tokens)
        woff = self.get_word_offsets(text) # (num_words,2) : character slices of words
        toff = offset_mapping # (num_tokens,2) : character slices of tokens
        wx1, wx2 = woff.T # (num_words,) x 2
        tx1, tx2 = toff.T # (num_tokens,) x 2
        ix1 = np.maximum(wx1[..., None], tx1[None, ...]) # (num_words,num_tokens)
        ix2 = np.minimum(wx2[..., None], tx2[None, ...]) # (num_words,num_tokens)
        ux1 = np.minimum(wx1[..., None], tx1[None, ...]) # (num_words,num_tokens)
        ux2 = np.maximum(wx2[..., None], tx2[None, ...]) # (num_words,num_tokens)
        ious = (ix2 - ix1).clip(min=0) / (ux2 - ux1 + 1e-12) # (num_words,num_tokens)
        assert (ious > 0).any(-1).all()

        word_boxes = []
        for row in ious: # ious.shape=(num_words,num_tokens)
            inds = row.nonzero()[0]
            word_boxes.append([inds[0], 0, inds[-1] + 1, 1])
        word_boxes = torch.FloatTensor(word_boxes)

        # word slices of ground truth spans
        gt_spans = []
        for _, row in df.iterrows():
            winds = row['predictionstring'].split()
            start = int(winds[0])
            end = int(winds[-1])
            span_label = TYPE2LABEL[row['discourse_type']]
            # gt_spans.append([start, end + 1, span_label])
            eff_label = row['discourse_effectiveness'] #ner_labels_eff
            gt_spans.append([start, end + 1, span_label, eff_label])

        gt_spans = torch.LongTensor(gt_spans)

        # random mask augmentation
        if np.random.random() < self.mask_prob:
            all_inds = np.arange(1, len(input_ids) - 1)
            n_mask = max(int(len(all_inds) * self.mask_ratio), 1)
            np.random.shuffle(all_inds)
            mask_inds = all_inds[:n_mask]
            input_ids[mask_inds] = self.tokenizer.mask_token_id

        return dict(
            text=text,
            text_id=text_id,
            input_ids=input_ids, # token ids
            attention_mask=attention_mask,
            word_boxes=word_boxes, # token slices of words
            gt_spans=gt_spans # word slices of ground truth spans
        )

    def strip_offset_mapping(self, text, offset_mapping):
        ret = []
        for start, end in offset_mapping:
            match = list(re.finditer('\S+', text[start:end]))
            if len(match) == 0:
                ret.append((start, end))
            else:
                span_start, span_end = match[0].span()
                ret.append((start + span_start, start + span_end))
        return np.array(ret)

    def get_word_offsets(self, text):
        matches = re.finditer("\S+", text)
        spans = []
        words = []
        for match in matches:
            span = match.span()
            word = match.group()
            spans.append(span)
            words.append(word)
        assert tuple(words) == tuple(text.split())
        return np.array(spans)



class CustomCollator(object):
    def __init__(self, tokenizer):
        self.pad_token_id = tokenizer.pad_token_id
        
    def __call__(self, samples):
        batch_size = len(samples)
        assert batch_size == 1, f'Only batch_size=1 supported, got batch_size={batch_size}.'

        sample = samples[0]

        max_seq_length = len(sample['input_ids'])
#         if self.attention_window is not None:
#             attention_window = self.attention_window
#             padded_length = (attention_window -
#                              max_seq_length % attention_window
#                              ) % attention_window + max_seq_length
#         else:
#             padded_length = max_seq_length
        padded_length = max_seq_length
        input_shape = (1, padded_length)
        input_ids = torch.full(input_shape,
                               self.pad_token_id,
                               dtype=torch.long)
        attention_mask = torch.zeros(input_shape, dtype=torch.long)

        seq_length = len(sample['input_ids'])
        input_ids[0, :seq_length] = sample['input_ids']
        attention_mask[0, :seq_length] = sample['attention_mask']

        text_id = sample['text_id']
        text = sample['text']
        word_boxes = sample['word_boxes']
        gt_spans = sample['gt_spans']

        return dict(text_id=text_id,
                    text=text,
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    word_boxes=word_boxes,
                    gt_spans=gt_spans)
    

In [20]:
def calc_overlap(row):
    """
    Calculates the overlap between prediction and
    ground truth and overlap percentages used for determining
    true positives.
    """
    set_pred = set(row.predictionstring_pred.split(" "))
    set_gt = set(row.predictionstring_gt.split(" "))
    # Length of each and intersection
    len_gt = len(set_gt)
    len_pred = len(set_pred)
    inter = len(set_gt.intersection(set_pred))
    overlap_1 = inter / len_gt
    overlap_2 = inter / len_pred
    return [overlap_1, overlap_2]


def score_feedback_comp_micro(pred_df, gt_df):
    """
    A function that scores for the kaggle
        Student Writing Competition
    Uses the steps in the evaluation page here:
        https://www.kaggle.com/c/feedback-prize-2021/overview/evaluation
    """
    gt_df = (
        gt_df[["id", "discourse_type", "predictionstring"]]
        .reset_index(drop=True)
        .copy()
    )
    pred_df = pred_df[["id", "class", "predictionstring"]].reset_index(drop=True).copy()
    pred_df["pred_id"] = pred_df.index
    gt_df["gt_id"] = gt_df.index
    # Step 1. all ground truths and predictions for a given class are compared.
    joined = pred_df.merge(
        gt_df,
        left_on=["id", "class"],
        right_on=["id", "discourse_type"],
        how="outer",
        suffixes=("_pred", "_gt"),
    )
    joined["predictionstring_gt"] = joined["predictionstring_gt"].fillna(" ")
    joined["predictionstring_pred"] = joined["predictionstring_pred"].fillna(" ")

    joined["overlaps"] = joined.apply(calc_overlap, axis=1)

    # 2. If the overlap between the ground truth and prediction is >= 0.5,
    # and the overlap between the prediction and the ground truth >= 0.5,
    # the prediction is a match and considered a true positive.
    # If multiple matches exist, the match with the highest pair of overlaps is taken.
    joined["overlap1"] = joined["overlaps"].apply(lambda x: eval(str(x))[0])
    joined["overlap2"] = joined["overlaps"].apply(lambda x: eval(str(x))[1])

    joined["potential_TP"] = (joined["overlap1"] >= 0.5) & (joined["overlap2"] >= 0.5)
    joined["max_overlap"] = joined[["overlap1", "overlap2"]].max(axis=1)
    tp_pred_ids = (
        joined.query("potential_TP")
        .sort_values("max_overlap", ascending=False)
        .groupby(["id", "predictionstring_gt"])
        .first()["pred_id"]
        .values
    )

    # 3. Any unmatched ground truths are false negatives
    # and any unmatched predictions are false positives.
    fp_pred_ids = [p for p in joined["pred_id"].unique() if p not in tp_pred_ids]

    matched_gt_ids = joined.query("potential_TP")["gt_id"].unique()
    unmatched_gt_ids = [c for c in joined["gt_id"].unique() if c not in matched_gt_ids]

    # Get numbers of each type
    TP = len(tp_pred_ids)
    FP = len(fp_pred_ids)
    FN = len(unmatched_gt_ids)
    # calc microf1
    my_f1_score = TP / (TP + 0.5 * (FP + FN))
    return my_f1_score


def score_feedback_comp(pred_df, gt_df, return_class_scores=False):
    class_scores = {}
    pred_df = pred_df[["id", "class", "predictionstring"]].reset_index(drop=True).copy()
    for discourse_type, gt_subset in gt_df.groupby("discourse_type"):
        pred_subset = (
            pred_df.loc[pred_df["class"] == discourse_type]
            .reset_index(drop=True)
            .copy()
        )
        class_score = score_feedback_comp_micro(pred_subset, gt_subset)
        class_scores[discourse_type] = class_score
    f1 = np.mean([v for v in class_scores.values()])
    if return_class_scores:
        return f1, class_scores
    return f1

In [ ]:
class TextSpanDetector_stage1(nn.Module):
    def __init__(self,
                 model_name,
                 tokenizer,
                 num_classes=7,
                dynamic_positive=True, #False,
                with_cp=False,
                learning_rate=1e-5,
                 head_learning_rate=1e-3,
                 num_train_steps=0,
                 hidden_dropout_prob=0, 
                 p_drop=0,
                 warmup_ratio = 0,
                 model_pretraining=None,
                 rnn='none',
                 loss='xentropy', #'mse',
                 head='simple',
                 msd='false',
                 multi_layers=1,
                 aug='none',
                 mixup_alpha=1.0,
                 aug_stop_epoch=999,
                 p_aug=0,
                 adv_sift='false',
                 l2norm='false',
                 s=30,
                 weight_decay=0.01,
                 freeze_layers='false',
                 mt='false',
                 w_mt=1,
                 scheduler='cosine',
                 num_cycles=1,
                 vb=False,
                ):
        super().__init__()
        ########################
        self._current_epoch = 1
        self.learning_rate = learning_rate
        self.head_learning_rate = head_learning_rate
        self.hidden_dropout_prob = hidden_dropout_prob
        self.warmup_ratio = warmup_ratio 
        self.num_train_steps = num_train_steps
        self.num_labels = 1 + 2 + num_classes
        self.tokenizer = tokenizer
        self.model_name = model_name
        self.loss = loss
        self.msd = msd
        self.multi_layers = multi_layers
        self.aug = aug
        self.mixup_alpha = mixup_alpha
        self.aug_stop_epoch = aug_stop_epoch
        self.p_aug = p_aug
        self.adv_sift = adv_sift
        self.l2norm = l2norm
        self.s = s
        self.weight_decay = weight_decay
        self.mt = mt
        self.w_mt = w_mt
        self.scheduler = scheduler
        self.num_cycles = num_cycles

        self.num_classes = num_classes
        self.dynamic_positive = dynamic_positive
        
        if model_pretraining is not None:
            self.transformer = model_pretraining.transformer
            self.config = model_pretraining.config
        else:
            self.config = AutoConfig.from_pretrained(model_name)
            self.config.update(
                {
                    "output_hidden_states": True,
                    "hidden_dropout_prob": self.hidden_dropout_prob,
                    "add_pooling_layer": False,
                    "num_labels": self.num_labels,
                }
            )
            self.transformer = AutoModel.from_pretrained(model_name, config=self.config)
            
        # resize
        self.transformer.resize_token_embeddings(len(tokenizer))
        
        if with_cp:
            self.transformer.gradient_checkpointing_enable()
            
        self.head = nn.Sequential(
            nn.Dropout(p_drop),
            nn.Linear(self.config.hidden_size, self.num_labels)
        )
        self._init_weights(self.head)
    
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            module.weight.data.normal_(mean=0.0, std=self.config.initializer_range)
            if module.bias is not None:
                module.bias.data.zero_()
        elif isinstance(module, nn.Embedding):
            module.weight.data.normal_(mean=0.0, std=self.config.initializer_range)
            if module.padding_idx is not None:
                module.weight.data[module.padding_idx].zero_()
        elif isinstance(module, nn.LayerNorm):
            module.bias.data.zero_()
            module.weight.data.fill_(1.0)
            
    def forward_logits(self, data):
        batch_size = data['input_ids'].size(0)
        assert batch_size == 1, f'Only batch_size=1 supported, got batch_size={batch_size}.'
        logits = self.transformer(input_ids=data['input_ids'],
                                  attention_mask=data['attention_mask']).last_hidden_state
        logits = self.head(logits)
        logits = aggregate_tokens_to_words(logits, data['word_boxes'])
        assert logits.size(0) == data['text'].split().__len__()

        obj_pred = logits[..., 0]
        reg_pred = logits[..., 1:3]
        cls_pred = logits[..., 3:]
        return obj_pred, reg_pred, cls_pred


    def predict(self, data, test_score_thr):
        data = to_gpu(data)
        obj_pred, reg_pred, cls_pred = self.forward_logits(data)
        obj_pred = obj_pred.sigmoid()
        reg_pred = reg_pred.exp()
        cls_pred = cls_pred.sigmoid()

        obj_scores = obj_pred
        cls_scores, cls_labels = cls_pred.max(-1)
        pr_scores = (obj_scores * cls_scores)**0.5
        pos_inds = pr_scores > test_score_thr

        if pos_inds.sum() == 0:
            return dict(text_id=data['text_id'])

        pr_score, pr_label = pr_scores[pos_inds], cls_labels[pos_inds]
        pos_loc = pos_inds.nonzero().flatten()
        start = pos_loc - reg_pred[pos_inds, 0]
        end = pos_loc + reg_pred[pos_inds, 1]

        min_idx, max_idx = 0, obj_pred.numel() - 1
        start = start.clamp(min=min_idx, max=max_idx).round().long()
        end = end.clamp(min=min_idx, max=max_idx).round().long()

        # nms
        keep = span_nms(start, end, pr_score)
        start = start[keep]
        end = end[keep]
        pr_score = pr_score[keep]
        pr_label = pr_label[keep]

        return dict(text_id=data['text_id'],
                    start=to_np(start),
                    end=to_np(end),
                    score=to_np(pr_score),
                    label=to_np(pr_label))
    
    
    def training_step(self, data, trn_df, test_score_thr):
        data = to_gpu(data)
        obj_pred, reg_pred, cls_pred = self.forward_logits(data)
        obj_target, reg_target, cls_target, pos_loc = self.build_target(data['gt_spans'], obj_pred, reg_pred, cls_pred)
        obj_loss, reg_loss, cls_loss = self.get_losses(obj_pred, reg_pred, cls_pred,
                                                       obj_target, reg_target, cls_target,
                                                       pos_loc)
        loss = obj_loss + reg_loss + cls_loss
        
        res = self.predict(data, test_score_thr)
        if len(res.keys())>1:
            pred = []
            text_id = res['text_id']
            for c,start,end in zip(res['label'],res['start'],res['end']):
                pred.append([text_id, LABEL2TYPE[c], ' '.join(np.arange(start,end+1).astype(str))])
            pred_df = pd.DataFrame(pred, columns=['id','class','predictionstring'])
            gt_df = trn_df[trn_df['id']==res['text_id']].reset_index(drop=True)
            score = score_feedback_comp(pred_df, gt_df, return_class_scores=False)
        else:
            score = 0
            
        return {
            'loss':loss,
            'obj_loss':obj_loss.item(),
            'reg_loss':reg_loss.item(),
            'cls_loss':cls_loss.item(),
            'score':score,
        }
    
    def validation_step(self, data, val_df, test_score_thr):
        data = to_gpu(data)
        obj_pred, reg_pred, cls_pred = self.forward_logits(data)
        obj_target, reg_target, cls_target, pos_loc = self.build_target(data['gt_spans'], obj_pred, reg_pred, cls_pred)
        obj_loss, reg_loss, cls_loss = self.get_losses(obj_pred, reg_pred, cls_pred,
                                                       obj_target, reg_target, cls_target,
                                                       pos_loc)
        
        loss = obj_loss + reg_loss + cls_loss
        
        res = self.predict(data, test_score_thr)
        gt_df = val_df[val_df['id']==res['text_id']].reset_index(drop=True)
        if len(res.keys())>1:
            pred = []
            text_id = res['text_id']
            for c, start, end in zip(res['label'],res['start'],res['end']):
                pred.append([
                    text_id, 
                    LABEL2TYPE[c], 
                    ' '.join(np.arange(start, end+1).astype(str)),
                    start, 
                    end
                    ])
            pred_df = pd.DataFrame(pred, columns=['id','class','predictionstring', 'start', 'end'])
        else:
            pred_df = pd.DataFrame([], columns=['id','class','predictionstring', 'start', 'end'])
            
        return {
            'data_id':res['text_id'],
            'loss':loss.item(),
            'obj_loss':obj_loss.item(),
            'reg_loss':reg_loss.item(),
            'cls_loss':cls_loss.item(),
            'pred_df':pred_df,
            'gt_df':gt_df,
        }
    
    def validation_epoch_end(self, outputs, add_epoch=True, return_class_scores=False):
        losses = []
        obj_losses = []
        reg_losses = []
        cls_losses = []
        pred_dfs = []
        gt_dfs = []
        for o in outputs:
            losses.append(o['loss'])
            obj_losses.append(o['obj_loss'])
            reg_losses.append(o['reg_loss'])
            cls_losses.append(o['cls_loss'])
            pred_dfs.append(o['pred_df'])
            gt_dfs.append(o['gt_df'])
        loss = np.array(losses).mean()
        obj_loss = np.array(obj_losses).mean()
        reg_loss = np.array(reg_losses).mean()
        cls_loss = np.array(cls_losses).mean()
        
        pred_df = pd.concat(pred_dfs, axis=0).reset_index(drop=True)
        gt_df = pd.concat(gt_dfs, axis=0).reset_index(drop=True)
        score = score_feedback_comp(pred_df, gt_df, return_class_scores=return_class_scores)
            
        if add_epoch:
            self._current_epoch += 1
            
        return {
            'loss':loss,
            'obj_loss':obj_loss,
            'reg_loss':reg_loss,
            'cls_loss':cls_loss,
            'score':score,
            'pred_df':pred_df,
        }
            

In [21]:
class TextSpanDetectorOriginal(nn.Module):
    """
     SpanDetector Original from: 
     https://github.com/Shujun-He/TeamSKT-Feedback-Prize---Predicting-Effective-Arguments-2nd-Place-solution/blob/main/Tom_solution/20_WoSpanDet/code/models_detector.py
    """
    def __init__(self, arch, num_classes=7, local_files_only=True):
        super().__init__()
        self.model = AutoModelForTokenClassification.from_pretrained(
            arch,
            num_labels=1 + 2 + num_classes,
            local_files_only=local_files_only
        )
        # self.tokenizer = AutoTokenizer.from_pretrained(arch, local_files_only=local_files_only)


# model_pretraining = TextSpanDetectorOriginal(args.model_name, num_classes=7)
# model_pretraining.load_state_dict(torch.load(args.pretrained_detector_path)["state_dict"])
# model_pretraining.model.deberta # 24 layers 
# model_pretraining.model.config

# ckp = torch.load("/mnt/md1/Work/dev_IME/Git_hacks/Feedback3/Feedback/checkpoints/tasckj/longformer_large_fold0.pth")["state_dict"]

In [48]:
class TextSpanDetector(nn.Module):
    def __init__(self,
                 model_name,
                 tokenizer,
                 num_classes=7,
                 num_classes2=2,
                 dynamic_positive=True, #False,
                 with_cp=False,
                 hidden_dropout_prob=0, 
                 learning_rate=1e-5,
                 head_learning_rate=1e-3,
                 num_train_steps=0,
                 p_drop=0,
                 warmup_ratio = 0,
                 model_pretraining=None,
                 rnn='none',
                 loss='mse',
                 head='simple',
                 msd='false',
                 multi_layers=1,
                 aug='none',
                 mixup_alpha=1.0,
                 aug_stop_epoch=999,
                 p_aug=0,
                 adv_sift='false',
                 l2norm='false',
                 s=30,
                 weight_decay=0.01,
                 mt='false',
                 w_mt=1,
                 scheduler='cosine',
                 num_cycles=1,
                 loss_weights = [1.0, 2.0], # [1.0, 1.0],
                 freeze_layers='true', ##'false',
                 vb=False,
                ):
        super().__init__()

        self.tokenizer = tokenizer
        self.model_name = model_name
        self.num_labels = 1 + 2 + num_classes
        self.num_labels2 = num_classes2
        self._current_epoch = 1
        self.learning_rate = learning_rate
        self.head_learning_rate = head_learning_rate
        self.hidden_dropout_prob = hidden_dropout_prob
        self.warmup_ratio = warmup_ratio 
        self.num_train_steps = num_train_steps
        self.loss = loss
        self.msd = msd
        self.multi_layers = multi_layers
        self.aug = aug
        self.mixup_alpha = mixup_alpha
        self.aug_stop_epoch = aug_stop_epoch
        self.p_aug = p_aug
        self.adv_sift = adv_sift
        self.l2norm = l2norm
        self.s = s
        self.weight_decay = weight_decay
        self.mt = mt
        self.w_mt = w_mt
        self.scheduler = scheduler
        self.num_cycles = num_cycles
        self.num_classes = num_classes
        self.dynamic_positive = dynamic_positive
        self.loss_weights = [1.0, 1.0]
        self.vb=vb

        if model_pretraining is not None:
            self.transformer = model_pretraining.model.deberta #model_pretraining.transformer
            self.config = model_pretraining.model.config
        else:
            self.config = AutoConfig.from_pretrained(model_name)
            self.config.update(
                {
                    "output_hidden_states": True,
                    "hidden_dropout_prob": self.hidden_dropout_prob,
                    "add_pooling_layer": False,
                    "num_labels": self.num_labels,
                }
            )
            self.transformer = AutoModel.from_pretrained(model_name, config=self.config)
            
        # resize
        self.transformer.resize_token_embeddings(len(tokenizer))
            
        #### freeze some layers for large models
        if freeze_layers == 'true':
            if 'deberta-v2-xxlarge' in model_name:
                print('freeze 24/48')
                self.transformer.embeddings.requires_grad_(False)
                self.transformer.encoder.layer[:24].requires_grad_(False) # freeze 24/48
            elif 'deberta-v2-xlarge' in model_name:
                print('freeze 12/24')
                self.transformer.embeddings.requires_grad_(False)
                self.transformer.encoder.layer[:12].requires_grad_(False) # freeze 12/24
            elif 'deberta-xlarge' in model_name:
                print('freeze 12/24')
                self.transformer.embeddings.requires_grad_(False)
                self.transformer.encoder.layer[:12].requires_grad_(False) # freeze 12/24
            elif 'funnel-transformer-xlarge' in model_name:
                self.transformer.embeddings.requires_grad_(False)
                self.transformer.encoder.blocks[:1].requires_grad_(False) # freeze 1/3

            ############
            elif 'deberta-v3-large' in model_name:
                self.transformer.embeddings.requires_grad_(False)

                print('freeze 12/24')
                self.transformer.encoder.layer[:12].requires_grad_(False) # freeze 12/24
                
                # print('freeze 24/48')
                # self.transformer.encoder.layer[:24].requires_grad_(False) # freeze 24/48

                # print('freeze all layers..')
                # self.transformer.encoder.layer[:].requires_grad_(False) 

            elif 'deberta-large' in model_name:
                self.transformer.embeddings.requires_grad_(False)
                print('freeze 12/24')
                self.transformer.encoder.layer[:12].requires_grad_(False) # freeze 12/24
                # print('freeze all layers..')
                # self.transformer.encoder.layer[:].requires_grad_(False) 
            
            elif 'deberta-base' in model_name:
                self.transformer.embeddings.requires_grad_(False)
                print('freeze half layers (6/12)..')
                self.transformer.encoder.layer[:6].requires_grad_(False) # freeze 12/24
                # print('freeze all layers (/12)..')
                # self.transformer.encoder.layer[:].requires_grad_(False) 
   

        #########################
        if with_cp:
            self.transformer.gradient_checkpointing_enable()
            
        self.head = nn.Sequential(
            nn.Dropout(p_drop),
            nn.Linear(self.config.hidden_size, self.num_labels)
        )
        self._init_weights(self.head)

        self.head2 = nn.Sequential(
            nn.Dropout(p_drop),
            nn.Linear(self.config.hidden_size, self.num_labels2)
        )
        self._init_weights(self.head2)
        
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            module.weight.data.normal_(mean=0.0, std=self.config.initializer_range)
            if module.bias is not None:
                module.bias.data.zero_()
        elif isinstance(module, nn.Embedding):
            module.weight.data.normal_(mean=0.0, std=self.config.initializer_range)
            if module.padding_idx is not None:
                module.weight.data[module.padding_idx].zero_()
        elif isinstance(module, nn.LayerNorm):
            module.bias.data.zero_()
            module.weight.data.fill_(1.0)
            
    
    #@torch.no_grad()
    def forward_logits(self, data):
        batch_size = data['input_ids'].size(0)
        assert batch_size == 1, f'Only batch_size=1 supported, got batch_size={batch_size}.'
        if self.vb: 
            print('[TRAIN] data word_boxes', data['word_boxes'].shape )
            print('[TRAIN] data input_ids', data['input_ids'].shape )
            print('[TRAIN] data gt_spans', data['gt_spans'].shape )

        outputs = self.transformer(input_ids=data['input_ids'], attention_mask=data['attention_mask'])
        x = outputs.last_hidden_state
        if self.vb: print('[TRAIN] hidden_state', x.shape, x )
        
        logits_tok = self.head(x)
        if self.vb: print('[TRAIN] logits type', logits_tok.shape )

        logits = aggregate_tokens_to_words(logits_tok, data['word_boxes'])
        assert logits.size(0) == data['text'].split().__len__()

        obj_pred = logits[..., 0]
        reg_pred = logits[..., 1:3]
        cls_pred = logits[..., 3:]

        return obj_pred, reg_pred, cls_pred, x


    def predict(self, data, test_score_thr):
        data = to_gpu(data)
        obj_pred, reg_pred, cls_pred, h = self.forward_logits(data)
        obj_pred = obj_pred.sigmoid()
        reg_pred = reg_pred.exp()
        cls_pred = cls_pred.sigmoid()

        obj_scores = obj_pred
        cls_scores, cls_labels = cls_pred.max(-1)
        pr_scores = (obj_scores * cls_scores)**0.5
        pos_inds = pr_scores > test_score_thr

        if pos_inds.sum() == 0:
            return dict(text_id=data['text_id'])

        pr_score, pr_label = pr_scores[pos_inds], cls_labels[pos_inds]
        pos_loc = pos_inds.nonzero().flatten()
        start = pos_loc - reg_pred[pos_inds, 0]
        end = pos_loc + reg_pred[pos_inds, 1]

        min_idx, max_idx = 0, obj_pred.numel() - 1
        start = start.clamp(min=min_idx, max=max_idx).round().long()
        end = end.clamp(min=min_idx, max=max_idx).round().long()

        # nms
        keep = span_nms(start, end, pr_score)
        start = start[keep]
        end = end[keep]
        pr_score = pr_score[keep]
        pr_label = pr_label[keep]

        return dict(text_id=data['text_id'],
                    start=to_np(start),
                    end=to_np(end),
                    score=to_np(pr_score),
                    label=to_np(pr_label))
    
    
    def training_step(self, data, trn_df, test_score_thr):
        data = to_gpu(data)

        obj_pred, reg_pred, cls_pred, h = self.forward_logits(data)
        if self.vb: 
            print('[TRAIN] obj_pred', len(obj_pred), reg_pred.shape, cls_pred.shape)


        obj_target, reg_target, cls_target, cls_target_eff, pos_loc = self.build_target(data['gt_spans'], obj_pred, reg_pred, cls_pred)
        obj_loss, reg_loss, cls_loss = self.get_losses(obj_pred, reg_pred, cls_pred,
                                                       obj_target, reg_target, cls_target,
                                                       pos_loc)
        loss1 = obj_loss + reg_loss + cls_loss
        
        res = self.predict(data, test_score_thr)
        #########################################
        h = h.squeeze(0) # seq_len x hidden_size
        if self.vb: print('[TRAIN] h:', h.shape, h)

        # start_, end_ = res['start'], res['end']
        start_, end_ = data['gt_spans'][:,0].tolist(), data['gt_spans'][:,1].tolist()

        pooled_outs = []
        for st_, en_ in zip(start_, end_):
            # active_logits_eff_word = res1['logits_eff_word'][st_:en_].mean()
            # active_logits_eff_word = self.head_eff(h[st_:en_+1])                   # len_span x 2 
            # active_logits_eff_word = active_logits_eff_word.mean(0).unsqueeze(0) # 1x2
            if self.vb:  print('[TRAIN] start/end', st_, en_)

            tmp = h[st_:en_+1,:] 
            if self.vb:  print('[TRAIN] tmp h', tmp.shape, tmp)
            tmp = tmp.mean(0)
            if self.vb:  print('[TRAIN] tmp h.mean(0)', tmp.shape, tmp)

            pooled_outs.append(tmp.unsqueeze(0))
        if self.vb: 
            print('[TRAIN] len pooled_outs', len(pooled_outs))

        pooled_outs = torch.cat(pooled_outs)
        if self.vb: print('[TRAIN] pooled_outs cat', pooled_outs.shape, pooled_outs)
        
        logits_eff = self.head2(pooled_outs)  
        if self.vb: print('[TRAIN] logits_eff', logits_eff.shape, logits_eff)

        discourse_eff_target = torch.tensor(data['gt_spans'][:,-1])
        if self.vb: print('[TRAIN] eff_target', discourse_eff_target.shape, discourse_eff_target)
        
        loss_fct = nn.CrossEntropyLoss(reduction='mean') 
        loss2 = loss_fct(logits_eff.softmax(-1), discourse_eff_target) 

        # 'sum'
        # loss_fct = nn.CrossEntropyLoss(reduction='sum')
        # cls_eff_loss = loss_fct(logits_eff.softmax(-1), discourse_eff_target) / num_total_samples
        ###############
        # Total loss
        loss = self.loss_weights[0]*loss1 + self.loss_weights[0]*loss2       
        
        logits_eff_softmax = logits_eff.softmax(-1)
        logits_eff_softmax = to_np(logits_eff_softmax)
        if self.vb: print('[TRAIN] logits_eff_softmax', logits_eff_softmax.shape)

        #########################################
        if len(res.keys())>1:
            pred = []
            text_id = res['text_id']
            for c,start,end, p_eff in zip(res['label'],res['start'],res['end'],logits_eff_softmax):
                pred.append([
                    text_id, 
                    LABEL2TYPE[c], 
                    ' '.join(np.arange(start,end+1).astype(str)),
                    c,
                    start, 
                    end,
                    p_eff[0], #logits_eff_softmax[ii,0], 
                    p_eff[1], #logits_eff_softmax[ii,1]  
                    ])
            pred_df = pd.DataFrame(pred, columns=['id','class','predictionstring', 'discourse_type', 'start', 'end', 'score_discourse_effectiveness_0', 'score_discourse_effectiveness_1'])
            gt_df = trn_df[trn_df['id']==res['text_id']].reset_index(drop=True)
            gt_df['discourse_type'] = gt_df['discourse_type'].map(TYPE2LABEL)

            # score_old = score_feedback_comp(pred_df, gt_df, return_class_scores=False)
            scores_dict = compute_scores(pred_df, gt_df, ids_selected=list(gt_df.id.unique()), threshold=0.5, return_segments=False, return_scores=False)
            score = scores_dict['F1_double_w'] 
            score_f1_sw = scores_dict['F1_single_w']
            score_f1_fp1 = scores_dict['F1_FB1']

        else:
            score = 0
            score_f1_sw = 0
            score_f1_fp1 = 0
            
        return {
            'loss':loss,
            'obj_loss':obj_loss.item(),
            'reg_loss':reg_loss.item(),
            'cls_loss':cls_loss.item(),
            'score':score,                #scores_dict['F1_double_w']
            'score_f1_sw':score_f1_sw,    #scores_dict['F1_single_w'], 
            'score_f1_fp1':score_f1_fp1,  #scores_dict['F1_FB1'], 
            ###
            'loss1':loss1,
            'loss2':loss2,
        }
    
    def validation_step(self, data, val_df, test_score_thr):
        data = to_gpu(data)
        obj_pred, reg_pred, cls_pred, h = self.forward_logits(data)
        obj_target, reg_target, cls_target, cls_target_eff, pos_loc = self.build_target(data['gt_spans'], obj_pred, reg_pred, cls_pred)
        obj_loss, reg_loss, cls_loss = self.get_losses(obj_pred, reg_pred, cls_pred,
                                                       obj_target, reg_target, cls_target,
                                                       pos_loc)
        
        loss1 = obj_loss + reg_loss + cls_loss
        
        res = self.predict(data, test_score_thr)
        #########################################
        h = h.squeeze(0) # seq_len x hidden_size

        # start_, end_ = res['start'], res['end']
        start_, end_ = data['gt_spans'][:,0].tolist(), data['gt_spans'][:,1].tolist()

        pooled_outs = []
        for st_, en_ in zip(start_, end_):
            pooled_outs.append(h[st_:en_+1].mean(0).unsqueeze(0))
        if self.vb: 
            print('[VAL] len pooled_outs', len(pooled_outs))

        pooled_outs = torch.cat(pooled_outs)
        logits_eff = self.head2(pooled_outs)  
        if self.vb: print('[VAL] logits_eff', logits_eff.shape)

        discourse_eff_target = torch.tensor(data['gt_spans'][:,-1])
        if self.vb: print('[VAL] eff_target', discourse_eff_target.shape)
        
        loss_fct = nn.CrossEntropyLoss(reduction='mean') 
        loss2 = loss_fct(logits_eff.softmax(-1), discourse_eff_target) 

        # 'sum'
        # loss_fct = nn.CrossEntropyLoss(reduction='sum')
        # cls_eff_loss = loss_fct(logits_eff.softmax(-1), discourse_eff_target) / num_total_samples
        ###############
        # Total loss
        loss = self.loss_weights[0]*loss1 + self.loss_weights[0]*loss2  
        ########

        logits_eff_softmax = logits_eff.softmax(-1) # num_disc x 2 
        logits_eff_softmax = to_np(logits_eff_softmax)

        ###############################################
        gt_df = val_df[val_df['id']==res['text_id']].reset_index(drop=True)
        if len(res.keys())>1:
            pred = []
            text_id = res['text_id']
            for ii, (c,start,end,p_eff) in enumerate(zip(res['label'],res['start'],res['end'],logits_eff_softmax)):
                pred.append([
                    text_id, 
                    LABEL2TYPE[c], 
                    ' '.join(np.arange(start,end+1).astype(str)),
                    c,
                    start, 
                    end,
                    p_eff[0], #logits_eff_softmax[ii,0], 
                    p_eff[1], #logits_eff_softmax[ii,1]  
                    ])
            pred_df = pd.DataFrame(pred, columns=['id','class','predictionstring', 'discourse_type', 'start', 'end', 'score_discourse_effectiveness_0', 'score_discourse_effectiveness_1'])
        else:
            pred_df = pd.DataFrame([], columns=['id','class','predictionstring', 'discourse_type', 'start', 'end', 'score_discourse_effectiveness_0', 'score_discourse_effectiveness_1'])
            
        return {
            'data_id':res['text_id'],
            'loss':loss.item(),
            'obj_loss':obj_loss.item(),
            'reg_loss':reg_loss.item(),
            'cls_loss':cls_loss.item(),
            'pred_df':pred_df,
            'gt_df':gt_df,
            ######
            'loss1':loss1.item(),
            'loss2':loss2.item(),
        }
    
    
    def validation_epoch_end(self, outputs, add_epoch=True):
        losses = []
        obj_losses = []
        reg_losses = []
        cls_losses = []
        eff_losses = []
        seg_losses = []
        pred_dfs = []
        gt_dfs = []
        for o in outputs:
            losses.append(o['loss'])
            obj_losses.append(o['obj_loss'])
            reg_losses.append(o['reg_loss'])
            cls_losses.append(o['cls_loss'])
            eff_losses.append(o['loss2'])
            seg_losses.append(o['loss1'])
            pred_dfs.append(o['pred_df'])
            gt_dfs.append(o['gt_df'])
        loss = np.array(losses).mean()
        obj_loss = np.array(obj_losses).mean()
        reg_loss = np.array(reg_losses).mean()
        cls_loss = np.array(cls_losses).mean()
        eff_loss = np.array(eff_losses).mean()
        seg_loss = np.array(seg_losses).mean()
        
        pred_df = pd.concat(pred_dfs, axis=0).reset_index(drop=True)
        gt_df   = pd.concat(gt_dfs, axis=0).reset_index(drop=True)
        gt_df['discourse_type'] = gt_df['discourse_type'].map(TYPE2LABEL)
        
        #score_old = score_feedback_comp(pred_df, gt_df, return_class_scores=False)
        scores_dict = compute_scores(pred_df, gt_df, ids_selected=list(gt_df.id.unique()), threshold=0.5, return_segments=False, return_scores=False)
        score = scores_dict['F1_double_w']
        # 'F1_FB1'   'F1_single_w'   'F1_double_w'  
    
        if add_epoch:
            self._current_epoch += 1
            
        return {
            'loss':loss,
            'obj_loss':obj_loss,
            'reg_loss':reg_loss,
            'cls_loss':cls_loss,
            'eff_loss':eff_loss,
            'loss1':seg_loss,
            'score':score,
            'score_f1_fp1':scores_dict['F1_FB1'],
            'score_f1_sw':scores_dict['F1_single_w'],
            ####
            'pred_df': pred_df,
            'gt_df': gt_df,
        }
    
    
    def configure_optimizers(self):
        optimizer = self.fetch_optimizer()
        scheduler = self.fetch_scheduler(optimizer)
        scheduler = {"scheduler": scheduler, "interval": "step", "frequency": 1}
        return [optimizer], [scheduler]
    
    def fetch_optimizer(self):
        head_params = list(self.head.named_parameters())
        param_optimizer = list(self.transformer.named_parameters())
        no_decay = ["bias", "LayerNorm.bias"]
        optimizer_parameters = [
            {
                "params": [p for n, p in param_optimizer if not any(nd in n for nd in no_decay)],
                "weight_decay": 0.01,
            },
            {
                "params": [p for n, p in param_optimizer if any(nd in n for nd in no_decay)],
                "weight_decay": 0.0,
            },
            {
                "params": [p for n,p in head_params], 
                "weight_decay": 0.01,
                "lr": self.head_learning_rate,
            },
        ]
        optimizer = AdamW(optimizer_parameters, lr=self.learning_rate)
        return optimizer

    def fetch_scheduler(self, optimizer):
        print('self.warmup_ratio = ', self.warmup_ratio)
        print('self.num_train_steps = ', self.num_train_steps)
        print('num_warmup_steps = ', int(self.warmup_ratio * self.num_train_steps))
        if self.scheduler=='cosine':
            scheduler = get_cosine_schedule_with_warmup(
                optimizer,
                num_warmup_steps=int(self.warmup_ratio * self.num_train_steps),
                num_training_steps=self.num_train_steps,
                num_cycles=0.5*self.num_cycles,
                last_epoch=-1,
            )
        elif self.scheduler=='cosine_hard':
            scheduler = get_cosine_with_hard_restarts_schedule_with_warmup(
                optimizer,
                num_warmup_steps=int(self.warmup_ratio * self.num_train_steps),
                num_training_steps=self.num_train_steps,
                num_cycles=self.num_cycles,
                last_epoch=-1,
            )
        elif self.scheduler=='linear':
            scheduler = get_linear_schedule_with_warmup(
                optimizer,
                num_warmup_steps=int(self.warmup_ratio * self.num_train_steps),
                num_training_steps=self.num_train_steps,
                last_epoch=-1,
            )
        return scheduler
    
    def get_losses(self, 
                   obj_pred, reg_pred, cls_pred, 
                   obj_target, reg_target, cls_target, 
                   pos_loc):
        
        num_total_samples = pos_loc.numel()
        assert num_total_samples > 0
        reg_pred = reg_pred[pos_loc].exp()
        reg_target = reg_target[pos_loc]
        px1 = pos_loc - reg_pred[:, 0]
        px2 = pos_loc + reg_pred[:, 1]
        gx1 = reg_target[:, 0]
        gx2 = reg_target[:, 1]
        ix1 = torch.max(px1, gx1)
        ix2 = torch.min(px2, gx2)
        ux1 = torch.min(px1, gx1)
        ux2 = torch.max(px2, gx2)
        inter = (ix2 - ix1).clamp(min=0)
        union = (ux2 - ux1).clamp(min=0) + 1e-12
        iou = inter / union
        
        reg_loss = -iou.log().sum() / num_total_samples
        
        cls_loss = F.binary_cross_entropy_with_logits(
            cls_pred[pos_loc],
            cls_target[pos_loc] * iou.detach().reshape(-1, 1),
            reduction='sum') / num_total_samples
        
        obj_loss = F.binary_cross_entropy_with_logits(
            obj_pred, 
            obj_target,
            reduction='sum') / num_total_samples
        
        return obj_loss, reg_loss, cls_loss

    @torch.no_grad()
    def build_target(self, gt_spans, obj_pred, reg_pred, cls_pred): #cls_pred2
        obj_target = torch.zeros_like(obj_pred)
        reg_target = torch.zeros_like(reg_pred)
        cls_target = torch.zeros_like(cls_pred)
        # cls_eff_target = torch.zeros_like(cls_pred2)
        cls_eff_target = torch.zeros((len(cls_pred), 2))
        
        # first token as positive
        pos_loc = gt_spans[:, 0]
        
        obj_target[pos_loc] = 1
        reg_target[pos_loc, 0] = gt_spans[:, 0].float()
        reg_target[pos_loc, 1] = gt_spans[:, 1].float()
        cls_target[pos_loc, gt_spans[:, 2]] = 1
        cls_eff_target[pos_loc, gt_spans[:, 3]] = 1

        # dynamically assign one more positive
        if self.dynamic_positive:
            cls_prob = (obj_pred.sigmoid().unsqueeze(1) *
                        cls_pred.sigmoid()).sqrt()
            
            for start, end, label, label_eff in gt_spans:
                _cls_prob = cls_prob[start:end]
                _cls_gt = _cls_prob.new_full((_cls_prob.size(0), ),
                                             label,
                                             dtype=torch.long)
                _cls_gt = F.one_hot(
                    _cls_gt, num_classes=_cls_prob.size(1)).type_as(_cls_prob)
                cls_cost = F.binary_cross_entropy(_cls_prob,
                                                  _cls_gt,
                                                  reduction='none').sum(-1)
                _reg_pred = reg_pred[start:end].exp()
                _reg_loc = torch.arange(_reg_pred.size(0),
                                        device=_reg_pred.device)
                px1 = _reg_loc - _reg_pred[:, 0]
                px2 = _reg_loc + _reg_pred[:, 1]
                ix1 = torch.max(px1, _reg_loc[0])
                ix2 = torch.min(px2, _reg_loc[-1])
                ux1 = torch.min(px1, _reg_loc[0])
                ux2 = torch.max(px2, _reg_loc[-1])
                inter = (ix2 - ix1).clamp(min=0)
                union = (ux2 - ux1).clamp(min=0) + 1e-12
                iou = inter / union
                iou_cost = -torch.log(iou + 1e-12)
                cost = cls_cost + iou_cost

                pos_ind = start + cost.argmin()
                obj_target[pos_ind] = 1
                reg_target[pos_ind, 0] = start
                reg_target[pos_ind, 1] = end
                cls_target[pos_ind, label] = 1
                cls_eff_target[pos_ind, label_eff] = 1

            pos_loc = (obj_target == 1).nonzero().flatten()
        return obj_target, reg_target, cls_target, cls_eff_target, pos_loc
    



In [24]:
# from models import TextSpanDetector, DatasetTrain, CustomCollator, TYPE2LABEL
# from awp import AWP
from transformers.models.deberta_v2 import DebertaV2TokenizerFast

In [53]:
def train_fold(args, trn_df, val_df, pseudo_df=None, fold=0):

    output_path = opj(args.output, args.exp_name, f"fold{str(args.fold)}")
    os.makedirs(output_path, exist_ok=True)
    
    if True:
        if 'deberta-v2' in args.model or 'deberta-v3' in args.model:
            tokenizer = DebertaV2TokenizerFast.from_pretrained(args.model, trim_offsets=False)
            # special_tokens_dict = {'additional_special_tokens': ['\n\n'] + [f'[{s.upper()}]' for s in list(TYPE2LABEL.keys())]}
            special_tokens_dict = {'additional_special_tokens': ['\n\n'] }
            _ = tokenizer.add_special_tokens(special_tokens_dict)
        else:
            tokenizer = AutoTokenizer.from_pretrained(args.model, trim_offsets=False)
            # special_tokens_dict = {'additional_special_tokens': [f'[{s.upper()}]' for s in list(TYPE2LABEL.keys())]}
            # _ = tokenizer.add_special_tokens(special_tokens_dict)

        trn_dataset = DatasetTrain(
            trn_df,
            tokenizer=tokenizer,
            mask_prob=args.mask_prob,
            mask_ratio=args.mask_ratio,
        )
        trn_dataloader = DataLoader(
            trn_dataset,
            batch_size=args.trn_batch_size,
            shuffle=True,
            collate_fn=CustomCollator(tokenizer),
            num_workers=4, 
            pin_memory=True,
            drop_last=True,
        )
        
        val_dataset = DatasetTrain(
            val_df,
            tokenizer=tokenizer,
            mask_prob=0,
            mask_ratio=0,
        )
        val_dataloader = DataLoader(
            val_dataset,
            batch_size=args.val_batch_size,
            shuffle=False,
            collate_fn=CustomCollator(tokenizer),
            num_workers=4, 
            pin_memory=True,
            drop_last=False,
        )
        
        #model
        num_trn_dataset = len(trn_dataset)
        num_train_steps = int(num_trn_dataset / args.trn_batch_size / args.accumulate_grad_batches * args.epochs)
        print('num_train_steps = ', num_train_steps)
        
        model_pretraining = None
        if 'tasckj' in args.pretrained_detector_path:
            # from models_detector import TextSpanDetectorOriginal
            model_pretraining = TextSpanDetectorOriginal(args.model_name, num_classes=7)
            model_pretraining.load_state_dict(torch.load(args.pretrained_detector_path)["state_dict"])
            print('Loaded encoder weights (taskj):', args.pretrained_detector_path)

        elif args.pretrained_detector_path != 'none':
            model_pretraining = TextSpanDetector_stage1(
                args.model,
                tokenizer,
                num_classes=args.num_classes,
                dynamic_positive=True,
                with_cp=False,
            )
            model_pretraining.load_state_dict(torch.load(args.pretrained_detector_path))
            print('Loaded encoder weights:', args.pretrained_detector_path)



        
        model = TextSpanDetector(
            args.model,
            tokenizer,
            num_classes=args.num_classes,
            dynamic_positive=True,
            with_cp=(args.check_pointing=='true'),
            hidden_dropout_prob=args.hidden_drop_prob, 
            p_drop=args.p_drop,
            scheduler=args.scheduler,
            learning_rate=args.lr,
            head_learning_rate=args.head_lr,
            num_train_steps=num_train_steps,
            warmup_ratio=args.warmup_ratio,
            model_pretraining=model_pretraining,
        )
        
        model = model.cuda()
        if args.pretrain_path != 'none':
            model.load_state_dict(torch.load(args.pretrain_path))
            print('Loaded weights:', args.pretrain_path)
            
        # Creates a GradScaler once at the beginning of training.
        scaler = torch.cuda.amp.GradScaler(enabled=(args.fp16=='true'))
        
        [optimizer], [scheduler] = model.configure_optimizers()
        scheduler = scheduler['scheduler']
        
        #training
        val_score_best  = -1e+99
        val_score_best2 = -1e+99
        val_loss_best   = 1e+99
        epoch_best = 0
        counter_ES = 0
        trn_score = 0
        start_time = time.time()
        for epoch in range(1, args.epochs+1):
            if epoch > args.stop_epoch:
                break
            if epoch < args.restart_epoch:
                scheduler.step()
                continue
                
            print('\nlr : ', [ group['lr'] for group in optimizer.param_groups ])
            
            #train
            trn_loss = 0
            trn_obj_loss = 0
            trn_reg_loss = 0
            trn_cls_loss = 0
            trn_eff_loss = 0
            trn_score = 0
            trn_score_f1_fp1 = 0
            trn_score_f1_sw = 0
            counter = 0
            tk0 = tqdm(trn_dataloader, total=int(len(trn_dataloader)))
            optimizer.zero_grad()
            trn_preds = []
            trn_trues = []
            
            if args.awp=='true':
                awp = AWP(
                    model,
                    criterion=nn.CrossEntropyLoss(reduction='none'), 
                    optimizer=optimizer,
                    apex=(args.fp16=='true'),
                    adv_lr=args.awp_lr, 
                    adv_eps=args.awp_eps
                )
            
            for i,data in enumerate(tk0):
                batch = trn_dataloader.batch_size
                model.train()
                with torch.cuda.amp.autocast(enabled=(args.fp16=='true')):
                    trn_res = model.training_step(data, trn_df, test_score_thr=args.test_score_thr)
                    loss = trn_res['loss']
                    
                    if args.accumulate_grad_batches > 1:
                        loss = loss / args.accumulate_grad_batches
                    scaler.scale(loss).backward()
                    grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), args.gradient_clip_val)
                    
                    trn_loss += loss.item() * batch * args.accumulate_grad_batches
                    trn_obj_loss += trn_res['obj_loss']  * batch
                    trn_reg_loss += trn_res['reg_loss']  * batch
                    trn_cls_loss += trn_res['cls_loss']  * batch
                    trn_score += trn_res['score'] * batch

                    trn_score_f1_fp1 += trn_res['score_f1_fp1'] * batch # scores_dict['F1_single_w'], 
                    trn_score_f1_sw  += trn_res['score_f1_sw'] * batch   # scores_dict['F1_FB1'], 
                             
                    if args.awp=='true':
                        if epoch >= args.awp_start_epoch:
                            loss = awp.attack_backward(data, trn_df, test_score_thr=args.test_score_thr)
                            if args.accumulate_grad_batches > 1:
                                loss = loss / args.accumulate_grad_batches
                            scaler.scale(loss).backward()
                            awp._restore()
                            trn_loss += loss.item() * batch * args.accumulate_grad_batches

                    if (i + 1) % args.accumulate_grad_batches == 0:
                        scaler.step(optimizer)
                        scaler.update()
                        optimizer.zero_grad()
                        scheduler.step()
                
                counter += 1
                tk0.set_postfix(loss=(trn_loss / (counter * batch) ), score=(trn_score / (counter * batch)), score_f1=(trn_score_f1_fp1 / (counter * batch)))
                
                # eval
                if args.eval_step!=-1 and (i+1)%args.eval_step==0:
                    model.eval()
                    outputs = []
                    for i,data in enumerate(val_dataloader):
                        with torch.no_grad():
                            outputs.append(model.validation_step(data, val_df, test_score_thr=args.test_score_thr))
                        #release GPU memory cache
                        del data
                        torch.cuda.empty_cache()
                        gc.collect()
                    
                    val_res = model.validation_epoch_end(outputs, add_epoch=False)
                    val_loss = val_res['loss']
                    val_score = val_res['score']
                    val_score_f1_fp1 = val_res['score_f1_fp1']
                    val_score_f1_sw = val_res['score_f1_sw']

                    #monitoring
                    print('\nepoch {:.0f}: trn_loss = {:.4f}, val_loss = {:.4f}, trn_score = {:.4f}, val_score = {:.4f}, val_score_f1_fp1 = {:.4f}, val_score_f1_sw = {:.4f}'.format(
                        epoch, 
                        trn_loss / (counter * trn_dataloader.batch_size), 
                        val_loss, 
                        trn_score / (counter * trn_dataloader.batch_size),
                        val_score, 
                        val_score_f1_fp1, 
                        val_score_f1_sw
                    ))
                    
                    print('trn_obj_loss={:.4f}, trn_reg_loss={:.4f}, trn_cls_loss={:.4f}, trn_eff_loss={:.4f}'.format(
                        trn_obj_loss / (counter * trn_dataloader.batch_size),
                        trn_reg_loss / (counter * trn_dataloader.batch_size), 
                        trn_cls_loss / (counter * trn_dataloader.batch_size),
                        trn_eff_loss / (counter * trn_dataloader.batch_size)
                    ))
                    
                    print('val_obj_loss={:.4f}, val_reg_loss={:.4f}, val_cls_loss={:.4f}, val_eff_loss={:.4f}'.format(
                        val_res['obj_loss'], val_res['reg_loss'], val_res['loss2']
                    ))
                          
                    # if args.slack_url!='none':
                    #     from utils import post_message
                    #     post_message(name='bot',
                    #                  message='epoch {:.0f}: trn_loss = {:.4f}, val_loss={:.4f}, trn_score = {:.4f}, val_score = {:.4f}'.format(
                    #                      epoch, trn_loss, val_loss, trn_score, val_score), 
                    #                  incoming_webhook_url=args.slack_url)
                    if args.early_stopping=='true':
                        if val_loss < val_loss_best: #val_score > val_score_best:
                            val_score_best = val_score #update
                            val_loss_best  = val_loss #update
                            epoch_best     = epoch #update
                            counter_ES     = 0 #reset
                            torch.save(model.state_dict(), opj(output_path,f'model_seed{args.seed}_fold{args.fold}_bestloss.pth')) #save
                            print('model (best loss) saved')
                        else:
                            counter_ES += 1
                        if counter_ES > args.patience:
                            print('early stopping, epoch_best {:.0f}, val_loss_best {:.5f}, val_score_best {:.5f}'.format(
                                epoch_best, val_loss_best, val_score_best))
                            break
                    else:
                        torch.save(model.state_dict(), opj(output_path,f'model_seed{args.seed}_fold{args.fold}_bestloss.pth')) #save

                    if val_score > val_score_best2:
                        val_score_best2 = val_score #update
                        torch.save(model.state_dict(), opj(output_path,f'model_seed{args.seed}_fold{args.fold}.pth')) #save
                        print('model (best score) saved')
                
            trn_loss  = trn_loss / len(trn_dataset)
            trn_score = trn_score / len(trn_dataset)
            
            #release GPU memory cache
            del data, loss
            torch.cuda.empty_cache()
            gc.collect()
            
            # save
            torch.save(model.state_dict(), opj(output_path,f'model_seed{args.seed}_fold{args.fold}_epoch{epoch}.pth')) #save

            #eval
            model.eval()
            outputs = []
            tk1 = tqdm(val_dataloader, total=int(len(val_dataloader)))
            for i,data in enumerate(tk1):
                with torch.no_grad():
                    outputs.append(model.validation_step(data, val_df, test_score_thr=args.test_score_thr))
                #release GPU memory cache
                del data
                torch.cuda.empty_cache()
                gc.collect()
            
            val_res = model.validation_epoch_end(outputs)
            # print(val_res.keys())
            # dict_keys(['loss', 'obj_loss', 'reg_loss', 'cls_loss', 'eff_loss', 'loss1', 'score', 'score_f1_fp1', 'score_f1_sw', 'pred_df', 'gt_df'])

            val_loss  = val_res['loss']
            val_score = val_res['score']
            val_score_f1_fp1 = val_res['score_f1_fp1']
            val_score_f1_sw  = val_res['score_f1_sw']
            
            #monitering
            print('epoch {:.0f}: trn_loss = {:.4f}, val_loss = {:.4f}, trn_score = {:.4f}, val_score = {:.4f}, val_score_f1_fp1 = {:.4f}, val_score_f1_sw = {:.4f}'.format(
                epoch, 
                trn_loss,
                val_loss,
                trn_score, 
                val_score,
                val_score_f1_fp1,
                val_score_f1_sw
            ))
            
            print('trn_obj_loss={:.4f}, trn_reg_loss={:.4f}, trn_cls_loss={:.4f}'.format(
                trn_obj_loss / len(trn_dataset),
                trn_reg_loss / len(trn_dataset), 
                trn_cls_loss / len(trn_dataset)
            ))
            
            print('val_obj_loss={:.4f}, val_reg_loss={:.4f}, val_cls_loss={:.4f}, val_eff_loss={:.4f}'.format(
                val_res['obj_loss'], val_res['reg_loss'], val_res['cls_loss'], val_res['eff_loss']
            ))
                  
            # if args.slack_url!='none':
            #     from utils import post_message
            #     post_message(name='bot',
            #                  message='epoch {:.0f}: trn_loss = {:.4f}, val_loss={:.4f}, trn_score = {:.4f}, val_score = {:.4f}'.format(
            #                      epoch, trn_loss, val_loss, trn_score, val_score), 
            #                  incoming_webhook_url=args.slack_url)
            if epoch%10 == 0: print(' elapsed_time = {:.1f} min'.format((time.time() - start_time)/60))
                
            if args.early_stopping=='true':
                if val_loss < val_loss_best: #val_score > val_score_best:
                    val_score_best = val_score #update
                    val_loss_best  = val_loss #update
                    epoch_best     = epoch #update
                    counter_ES     = 0 #reset
                    torch.save(model.state_dict(), opj(output_path,f'model_seed{args.seed}_fold{args.fold}_bestloss.pth')) #save
                    print('model (best loss) saved')
                else:
                    counter_ES += 1
                if counter_ES > args.patience:
                    print('early stopping, epoch_best {:.0f}, val_loss_best {:.5f}, val_score_best {:.5f}'.format(
                        epoch_best, val_loss_best, val_score_best))
                    break
            else:
                torch.save(model.state_dict(), opj(output_path, f'model_seed{args.seed}_fold{args.fold}_bestloss.pth')) #save
               
            if val_score > val_score_best2:
                val_score_best2 = val_score #update
                torch.save(model.state_dict(), opj(output_path, f'model_seed{args.seed}_fold{args.fold}.pth')) #save
                print('model (best score) saved')
                
        #best model
        if args.early_stopping=='true' and counter_ES<=args.patience:
            print('epoch_best {:d}, val_loss_best {:.5f}, val_score_best {:.5f}'.format(epoch_best, val_loss_best, val_score_best))
        
        del model
        torch.cuda.empty_cache()
        gc.collect()



In [26]:
def eval_fold(args, trn_df, val_df, fold=0):

    output_path = opj(args.output, args.exp_name, f"fold{str(args.fold)}")
    os.makedirs(output_path, exist_ok=True)
    
    if True:
        if 'deberta-v2' in args.model or 'deberta-v3' in args.model:
            from transformers.models.deberta_v2 import DebertaV2TokenizerFast
            tokenizer = DebertaV2TokenizerFast.from_pretrained(args.model, trim_offsets=False)
            special_tokens_dict = {'additional_special_tokens': ['\n\n'] + [f'[{s.upper()}]' for s in list(TYPE2LABEL.keys())]}
            _ = tokenizer.add_special_tokens(special_tokens_dict)
        else:
            tokenizer = AutoTokenizer.from_pretrained(args.model, trim_offsets=False)
            special_tokens_dict = {'additional_special_tokens': [f'[{s.upper()}]' for s in list(TYPE2LABEL.keys())]}
            _ = tokenizer.add_special_tokens(special_tokens_dict)

        trn_dataset = DatasetTrain(
            trn_df,
            tokenizer=tokenizer,
            mask_prob=args.mask_prob,
            mask_ratio=args.mask_ratio,
        )
        trn_dataloader = DataLoader(
            trn_dataset,
            batch_size=args.trn_batch_size,
            shuffle=True,
            collate_fn=CustomCollator(tokenizer),
            num_workers=4, 
            pin_memory=True,
            drop_last=True,
        )
        
        val_dataset = DatasetTrain(
            val_df,
            tokenizer=tokenizer,
            mask_prob=0,
            mask_ratio=0,
        )
        val_dataloader = DataLoader(
            val_dataset,
            batch_size=args.val_batch_size,
            shuffle=False,
            collate_fn=CustomCollator(tokenizer),
            num_workers=4, 
            pin_memory=True,
            drop_last=False,
        )
        
        # #model
        num_trn_dataset = len(trn_dataset)
        num_train_steps = int(num_trn_dataset / args.trn_batch_size / args.accumulate_grad_batches * args.epochs)
        print('num_train_steps = ', num_train_steps)
        # model_pretraining = None
        
        
        model = TextSpanDetector(
            args.model,
            tokenizer,
            num_classes=args.num_classes,
            dynamic_positive=True,
            with_cp=(args.check_pointing=='true'),
            hidden_dropout_prob=args.hidden_drop_prob, 
            p_drop=args.p_drop,
            learning_rate=args.lr,
            head_learning_rate=args.head_lr,
            num_train_steps=num_train_steps,
            warmup_ratio=args.warmup_ratio,
            model_pretraining=None,
            rnn=args.rnn,
            loss=args.loss,
            # head=args.head,
            scheduler=args.scheduler,
            # msd=args.msd,
            # multi_layers=args.multi_layers,
            # aug=args.aug,
            # mixup_alpha=args.mixup_alpha,
            # aug_stop_epoch=args.aug_stop_epoch,
            # p_aug=args.p_aug,
            # adv_sift=args.adv_sift,
            # l2norm=args.l2norm,
            # weight_decay=args.weight_decay,
            # freeze_layers=args.freeze_layers,
            # mt=args.mt,
            # w_mt=args.w_mt,
            # num_cycles=args.num_cycles,
        )
        
        model = model.cuda()
        if args.pretrain_path != 'none':
            model.load_state_dict(torch.load(args.pretrain_path))
            
        # Creates a GradScaler once at the beginning of training.
        scaler = torch.cuda.amp.GradScaler(enabled=(args.fp16=='true'))
        
        [optimizer], [scheduler] = model.configure_optimizers()
        scheduler = scheduler['scheduler']
        
        # #training
        # val_score_best  = -1e+99
        # val_score_best2 = -1e+99
        # val_loss_best   = 1e+99
        # epoch_best = 0
        # counter_ES = 0
        # trn_score = 0
        # start_time = time.time()
        
        #eval
        model.eval()
        outputs = []
        tk1 = tqdm(val_dataloader, total=int(len(val_dataloader)))
        for i,data in enumerate(tk1):
            with torch.no_grad():
                outputs.append(model.validation_step(data, val_df, test_score_thr=args.test_score_thr))
            #release GPU memory cache
            del data
            torch.cuda.empty_cache()
            gc.collect()
        #######
        val_res = model.validation_epoch_end(outputs)
        val_loss, val_score = val_res['loss'], val_res['score']
        
        print('Fold {:.0f}:   val_loss = {:.4f}, val_score = {:.4f}'.format(fold, val_loss, val_score))
        
        print('val_obj_loss={:.4f}, val_reg_loss={:.4f}, val_cls_loss={:.4f}'.format(
            val_res['obj_loss'], val_res['reg_loss'], val_res['cls_loss']
        ))
                  
       
        del model
        torch.cuda.empty_cache()
        gc.collect()

    return val_res['pred_df']

In [27]:
# CFG.model_name = 'microsoft/deberta-v3-large'
# CFG.model_name = 'microsoft/deberta-large'
CFG.model_name 

'microsoft/deberta-base'

In [28]:
args = CFG 

args.fold = 0
args.seed = 100
args.version = '1'
args.model = CFG.model_name 
args.lr = 6e-5 
args.head_lr = 6e-5 
args.warmup_ratio = 0.05
args.epochs = 5
args.trn_batch_size = 1
args.val_batch_size = 1
args.mask_prob = 0.7
args.mask_ratio = 0.25
args.accumulate_grad_batches = 4
args.gradient_clip_val = 1.0 
args.num_classes = 7
args.check_pointing = True
args.hidden_drop_prob = 0.1
args.p_drop = 0.0
args.loss = 'xentropy'
args.pretrain_path = 'none'
args.early_stopping = 'false'
args.patience = 100
args.slack_url = 'none'
args.eval_step = -1
args.stop_epoch = 5
args.test_score_thr = 0.4 #0.5
args.fp16 = 'false' #'true'


# args.pretrained_detector_path = 'none'
# args.pretrained_detector_path = "/mnt/md1/Work/dev_IME/Git_hacks/Feedback3/output/exp50_d3b/fold0/model_seed100_fold0_epoch5.pth"
# args.pretrained_detector_path = "/mnt/md1/Work/dev_IME/Git_hacks/Feedback3/Feedback/checkpoints/tasckj/deberta_large_fold0.pth"

args.pretrain_path = 'none'
# args.pretrain_path = "/mnt/md1/Work/dev_IME/Git_hacks/Feedback3/output/exp50_d3b/fold0/model_seed100_fold0_epoch5.pth"

print(args.pretrained_detector_path)

/mnt/md1/Work/dev_IME/Git_hacks/Feedback3/Feedback/checkpoints/tasckj/deberta_base_fold0.pth


In [54]:
# args.pretrain_path = 'none'

### resume from epoch 1
args.pretrain_path = '/mnt/md1/Work/dev_IME/Git_hacks/Feedback3/output/exp61_V7b/fold0/model_seed100_fold0_epoch1_6605.pth'

In [55]:
args.epochs = 4 #5

In [29]:
train_df, valid_df = get_data(fold=args.fold)

print('train.shape:', train_df.shape,'\t','test.shape:', valid_df.shape)

(144289, 16)


,essay_id_comp,discourse_id,discourse_start,discourse_end,discourse_type,predictionstring,discourse_text,discourse_effectiveness,discourse_type_num,source_text,fold,essay_id,id,discourse_effectiveness_int,essay_text,class
0,423A1CA112E2,1.622628e+12,8,229,Lead,1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 1...,Modern humans today are always on their phone....,Non-Effective,Lead 1,NaN,4,423A1CA112E2,423A1CA112E2,0,Phones\n\nModern humans today are always on th...,Lead
1,423A1CA112E2,1.622628e+12,230,312,Position,45 46 47 48 49 50 51 52 53 54 55 56 57 58 59,They are some really bad consequences when stu...,Non-Effective,Position 1,NaN,4,423A1CA112E2,423A1CA112E2,0,Phones\n\nModern humans today are always on th...,Position
2,423A1CA112E2,1.622628e+12,313,400,Evidence,60 61 62 63 64 65 66 67 68 69 70 71 72 73 74 75,Some certain areas in the United States ban ph...,Non-Effective,Evidence 1,NaN,4,423A1CA112E2,423A1CA112E2,0,Phones\n\nModern humans today are always on th...,Evidence
3,423A1CA112E2,1.622628e+12,401,756,Evidence,76 77 78 79 80 81 82 83 84 85 86 87 88 89 90 9...,"When people have phones, they know about certa...",Non-Effective,Evidence 2,NaN,4,423A1CA112E2,423A1CA112E2,0,Phones\n\nModern humans today are always on th...,Evidence
4,423A1CA112E2,1.622628e+12,757,884,Claim,139 140 141 142 143 144 145 146 147 148 149 15...,Driving is one of the way how to get around. P...,Non-Effective,Claim 1,NaN,4,423A1CA112E2,423A1CA112E2,0,Phones\n\nModern humans today are always on th...,Claim


train.shape: (115427, 16) 	 test.shape: (28862, 16)


In [30]:
train_df['discourse_effectiveness_str'] = train_df['discourse_effectiveness']
train_df['discourse_effectiveness'] = train_df['discourse_effectiveness'].map(ner_labels_eff)

valid_df['discourse_effectiveness_str'] = valid_df['discourse_effectiveness']
valid_df['discourse_effectiveness'] = valid_df['discourse_effectiveness'].map(ner_labels_eff)

train_df['discourse_effectiveness'].value_counts()

discourse_effectiveness
0    95763
1    19664
Name: count, dtype: int64

In [47]:
def check_model():
## if True:

    args.pretrain_path = "/mnt/md1/Work/dev_IME/Git_hacks/Feedback3/output/exp61_V7b/fold0/model_seed100_fold0_epoch1.pth"

    if 'deberta-v2' in args.model or 'deberta-v3' in args.model:
        tokenizer = DebertaV2TokenizerFast.from_pretrained(args.model, trim_offsets=False)
        special_tokens_dict = {'additional_special_tokens': ['\n\n'] + [f'[{s.upper()}]' for s in list(TYPE2LABEL.keys())]}
        _ = tokenizer.add_special_tokens(special_tokens_dict)
    else:
        tokenizer = AutoTokenizer.from_pretrained(args.model, trim_offsets=False)
        special_tokens_dict = {'additional_special_tokens': [f'[{s.upper()}]' for s in list(TYPE2LABEL.keys())]}
        _ = tokenizer.add_special_tokens(special_tokens_dict)
    
    # ds = DatasetTrain(train_df, tokenizer)
    ds = DatasetTrain(valid_df, tokenizer)
    dl = DataLoader(ds, batch_size=1, collate_fn=CustomCollator(tokenizer),)

    model_pretraining = None
    if 'tasckj' in args.pretrained_detector_path:
        model_pretraining = TextSpanDetectorOriginal(args.model_name, num_classes=7, ) #local_files_only=False
        model_pretraining.load_state_dict(torch.load(args.pretrained_detector_path)["state_dict"])
        print('Loaded encoder weights (taskj):', args.pretrained_detector_path)


    model = TextSpanDetector(
                args.model,
                tokenizer,
                num_classes=args.num_classes,
                dynamic_positive=True,
                with_cp=(args.check_pointing=='true'),
                hidden_dropout_prob=args.hidden_drop_prob, 
                p_drop=args.p_drop,
                learning_rate=args.lr,
                head_learning_rate=args.head_lr,
                warmup_ratio=args.warmup_ratio,
                model_pretraining=model_pretraining,
                scheduler=args.scheduler,
                # vb=True
            )
    model.transformer.resize_token_embeddings(len(tokenizer))
    model = model.cuda()
    if args.pretrain_path != 'none':
        model.load_state_dict(torch.load(args.pretrain_path))
        print('Loaded weights from:', args.pretrain_path)

    batch = next(iter(dl))
    # out = model.training_step(batch, train_df, 0.4)

    model.eval()
    # out = model.validation_step(batch, valid_df, 0.4)
    
    outputs = []
    tk1 = tqdm(dl, total=int(len(dl)))
    for i,data in enumerate(tk1):
        with torch.no_grad():
            outputs.append(model.validation_step(data, valid_df, test_score_thr=args.test_score_thr))
        #release GPU memory cache
        del data
        torch.cuda.empty_cache()
        gc.collect()
    
    val_res = model.validation_epoch_end(outputs)
    print(val_res.keys())
    print()

    # print(out.keys())
    for k,v in val_res.items():
        if k not in ["text", "text_id"]:
            print(k, v)

    del model 
    del outputs
    gc.collect()

# # #####
# check_model()

Loaded 3120 samples.


Some weights of the model checkpoint at microsoft/deberta-base were not used when initializing DebertaForTokenClassification: ['lm_predictions.lm_head.dense.weight', 'lm_predictions.lm_head.LayerNorm.weight', 'lm_predictions.lm_head.dense.bias', 'deberta.embeddings.position_embeddings.weight', 'lm_predictions.lm_head.LayerNorm.bias', 'lm_predictions.lm_head.bias']
- This IS expected if you are initializing DebertaForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DebertaForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of DebertaForTokenClassification were not initialized from the model checkpoint at microsoft/deberta-base and are newly initial

Loaded encoder weights (taskj): /mnt/md1/Work/dev_IME/Git_hacks/Feedback3/Feedback/checkpoints/tasckj/deberta_base_fold0.pth
freeze half layers (6/12)..


Token indices sequence length is longer than the specified maximum sequence length for this model (621 > 512). Running this sequence through the model will result in indexing errors


Loaded weights from: /mnt/md1/Work/dev_IME/Git_hacks/Feedback3/output/exp61_V7b/fold0/model_seed100_fold0_epoch1.pth


100%|██████████| 3120/3120 [19:03<00:00,  2.73it/s]


dict_keys(['loss', 'obj_loss', 'reg_loss', 'cls_loss', 'eff_loss', 'loss1', 'score', 'score_f1_fp1', 'score_f1_sw', 'pred_df', 'gt_df'])

loss 2.7573931001699887
obj_loss 1.2014499258094777
reg_loss 0.3182159728472097
cls_loss 0.771732841947904
eff_loss 0.4659943567732206
loss1 2.2913987417156116
score 0.0
score_f1_fp1 0.0
score_f1_sw 0.0
pred_df                  id         class  \
0      001552828BD0      Evidence   
1      001552828BD0      Evidence   
2      001552828BD0          Lead   
3      001552828BD0      Evidence   
4      001552828BD0      Evidence   
...             ...           ...   
27722  FFF1442D6698         Claim   
27723  FFF1442D6698  Counterclaim   
27724  FFF1442D6698      Evidence   
27725  FFF1442D6698      Rebuttal   
27726  FFF1442D6698      Evidence   

                                        predictionstring  discourse_type  \
0      302 303 304 305 306 307 308 309 310 311 312 31...               3   
1      197 198 199 200 201 202 203 204 205 206 207 20.

In [ ]:
# epoch 1: trn_loss = 3.4917, val_loss = 2.7503, trn_score = 0.6005, val_score = 0.6605, val_score_f1_fp1 = 0.7442, val_score_f1_sw = 0.7008
# trn_obj_loss=1.5610, trn_reg_loss=0.4492, trn_cls_loss=1.0130

In [56]:
### run1 freeze backbone 12/24

train_fold(args, train_df, valid_df, pseudo_df=None, fold=args.fold)

Loaded 12474 samples.
Loaded 3120 samples.
num_train_steps =  12474


Some weights of the model checkpoint at microsoft/deberta-base were not used when initializing DebertaForTokenClassification: ['lm_predictions.lm_head.dense.weight', 'lm_predictions.lm_head.LayerNorm.weight', 'lm_predictions.lm_head.dense.bias', 'deberta.embeddings.position_embeddings.weight', 'lm_predictions.lm_head.LayerNorm.bias', 'lm_predictions.lm_head.bias']
- This IS expected if you are initializing DebertaForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DebertaForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of DebertaForTokenClassification were not initialized from the model checkpoint at microsoft/deberta-base and are newly initial

Loaded encoder weights (taskj): /mnt/md1/Work/dev_IME/Git_hacks/Feedback3/Feedback/checkpoints/tasckj/deberta_base_fold0.pth
freeze half layers (6/12)..
Loaded weights: /mnt/md1/Work/dev_IME/Git_hacks/Feedback3/output/exp61_V7b/fold0/model_seed100_fold0_epoch1_6605.pth
self.warmup_ratio =  0.05
self.num_train_steps =  12474
num_warmup_steps =  623

lr :  [0.0, 0.0, 0.0]


  0%|          | 0/12474 [00:00<?, ?it/s]Token indices sequence length is longer than the specified maximum sequence length for this model (538 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (527 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (540 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (684 > 512). Running this sequence through the model will result in indexing errors
  0%|          | 0/3120 [00:00<?, ?it/s]Token indices sequence length is longer than the specified maximum sequence length for this model (621 > 512). Running this sequence through the model will result in indexing errors
Token ind

epoch 1: trn_loss = 2.8843, val_loss = 2.6968, trn_score = 0.6555, val_score = 0.6789, val_score_f1_fp1 = 0.7639, val_score_f1_sw = 0.7211
trn_obj_loss=1.2497, trn_reg_loss=0.3405, trn_cls_loss=0.8299
val_obj_loss=1.1590, val_reg_loss=0.2971, val_cls_loss=0.7748, val_eff_loss=0.4660
model (best score) saved

lr :  [5.3673958191869564e-05, 5.3673958191869564e-05, 5.3673958191869564e-05]


  0%|          | 0/12474 [00:00<?, ?it/s]Token indices sequence length is longer than the specified maximum sequence length for this model (554 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (869 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (566 > 512). Running this sequence through the model will result in indexing errors
 70%|███████   | 8775/12474 [28:51<12:52,  4.79it/s, loss=2.78, score=0.673, score_f1=0.752]  

In [ ]:
# epoch 1: trn_loss = 4.1163, val_loss = 3.0400, trn_score = 0.3931, val_score = 0.6105
# trn_obj_loss=2.2601, trn_reg_loss=0.5893, trn_cls_loss=1.2669
# val_obj_loss=1.5935, val_reg_loss=0.4368, val_cls_loss=1.0098

In [ ]:
# args.pretrain_path = "/mnt/md1/Work/dev_IME/Git_hacks/Feedback3/output/exp50_d3b/fold0/model_seed100_fold0_epoch5.pth"

# val_preds = eval_fold(args, train_df, valid_df, fold=args.fold)

# val_preds.to_csv(f"/mnt/md1/Work/dev_IME/Git_hacks/Feedback3/output/exp50_d3b/val_pred_f{args.fold}.csv", index=False)

In [ ]:
# val_preds

## Stage 2

In [ ]:
# https://github.com/Shujun-He/TeamSKT-Feedback-Prize---Predicting-Effective-Arguments-2nd-Place-solution/blob/main/Tom_solution/20_WoSpanDet/exp/20_v1_01.ipynb'

VERSION = '20_v1_01'
# FOLD_PATH = '../../00_EDA/00_v2_07/result/'
MODEL = 'microsoft/deberta-large'
LR = 8e-6 
HEAD_LR = 8e-6 
SEED = 100
TRN_BS = 1
VAL_BS = 1
ACCUM_STEP = 1
EPOCHS = 5 
STOP_EPOCH = 5 
RESTART = 1
HIDDEN_DROP_PROB = 0
P_DROP = 0
RNN = 'none'
WARMUP_RATIO = 0.1
LOSS = 'xentropy'
HEAD = 'simple'
AUG = 'false' 
MIXUP_ALPHA = 1.0
P_AUG = 0
AUG_STOP_EPOCH = 2
MSD = 'true'
MULTI_LAYERS = 1
EVAL_STEP = -1 
NUM_LABELS = 3
NUM_LABELS_2 = 7
ADV_SIFT = 'false'
FP16 = 'true'
WD = 0.01
FREEZE = 'false'
MULTI_TASK = 'false'
W_MT = 1.0 
AWP = 'true'
AWP_LR = 1e-2 
AWP_EPS = 1e-3 
AWP_START_EPOCH = 1
PRETRAINED_DETECTOR_PATH = 'none'
MASK_PROB = 0.8
MASK_RATIO = 0.3
SCHEDULER = 'cosine_hard'
NUM_CYCLES = EPOCHS
CP = 'false'
WINDOW_SIZE = 512
INNER_LEN = 384
EDGE_LEN = 64


In [ ]:
class WoSpanDetector(nn.Module):
    def __init__(self, 
                 model_name, 
                 tokenizer,
                 num_labels=7, 
                 num_labels_2=2,
                 hidden_dropout_prob=0, 
                 learning_rate=LR, #1e-5,
                 head_learning_rate=1e-3,
                 num_train_steps=0,
                 p_drop=0,
                 warmup_ratio = 0,
                 model_pretraining=None,
                 rnn='none',
                 loss=LOSS,
                 head=HEAD,
                 msd='false',
                 multi_layers=1,
                 aug='none',
                 mixup_alpha=1.0,
                 aug_stop_epoch=999,
                 p_aug=0.5,
                 adv_sift='false',
                 l2norm='false',
                 s=30,
                 weight_decay=0.01,
                 freeze_layers='false',
                 mt='true', #'false',
                 w_mt=1.0, #0.5,
                 scheduler='cosine',
                 num_cycles=1,
                 with_cp=False,
                 window_size=-100,
                 inner_len=-100,
                 edge_len=-100,
                 adam_bits=32,
                 ###
                # pretrained_path="",
                # pretrained_path="/mnt/md1/Work/dev_IME/Git_hacks/Feedback3/Feedback/checkpoints/", ## None
                use_sigmoid=False,
                nms_thr=0.5,
                test_score_thr=0.4, # 0.5
                dynamic_positive=True, #False,
                local_files_only=False,
                init_cfg=None, 
                train_df = None,
                valid_df = None,
                predict_df = None,
                ###
                 device='cuda',
                 cfg=None,
                 **kwargs,
                ):
        super().__init__()
        self.cfg=cfg
        self._current_epoch = 1
        self.learning_rate = learning_rate
        self.head_learning_rate = head_learning_rate
        self.hidden_dropout_prob = hidden_dropout_prob
        self.warmup_ratio = warmup_ratio 
        self.num_train_steps = num_train_steps
        self.num_labels = num_labels+1+2
        self.num_labels_2 = num_labels_2
        self.tokenizer = tokenizer
        self.model_name = model_name
        self.loss = loss
        self.msd = msd
        self.multi_layers = multi_layers
        self.aug = aug
        self.mixup_alpha = mixup_alpha
        self.aug_stop_epoch = aug_stop_epoch
        self.p_aug = p_aug
        self.adv_sift = adv_sift
        self.l2norm = l2norm
        self.s = s
        self.weight_decay = weight_decay
        self.w_mt = w_mt
        self.scheduler = scheduler
        self.num_cycles = num_cycles
        self.last_hidden_only = False
        self.window_size = window_size
        self.inner_len = inner_len
        self.edge_len = edge_len
        self.adam_bits = adam_bits
        self.mt = mt
        self.dynamic_positive = dynamic_positive
        self.nms_thr = nms_thr
        self.device = device
        self.use_sigmoid=use_sigmoid
        self.best_score = 0.0
        self.test_score_thr = test_score_thr
        self.val_outputs = []

        ### ----------------------------
        ### BACKBONE
        ### ----------------------------

        if model_pretraining is not None:
            try:
                self.transformer = model_pretraining.transformer
                self.config = model_pretraining.config
            except:
                self.last_hidden_only = True
                self.multi_layers = 1
                self.transformer = model_pretraining.model.deberta
                self.config = AutoConfig.from_pretrained(model_name)
                self.config.update(
                    {
                        "output_hidden_states": True,
                        "hidden_dropout_prob": self.hidden_dropout_prob,
                        "add_pooling_layer": False,
                        "num_labels": self.num_labels+1+2,
                    }
                )
        else:
            self.config = AutoConfig.from_pretrained(model_name)
            self.config.update(
                {
                    "output_hidden_states": True,
                    "hidden_dropout_prob": self.hidden_dropout_prob,
                    "add_pooling_layer": False,
                    "num_labels": self.num_labels+1+2,
                }
            )
            self.transformer = AutoModel.from_pretrained(model_name, config=self.config)
            
        # resize
        self.transformer.resize_token_embeddings(len(tokenizer))
        
        if self.cfg.gradient_checkpointing:
            self.transformer.gradient_checkpointing_enable()
            
        # freeze some layers for large models
        if freeze_layers == 'true':
            if 'deberta-v2-xxlarge' in model_name:
                print('freeze 24/48')
                self.transformer.embeddings.requires_grad_(False)
                self.transformer.encoder.layer[:24].requires_grad_(False) # freeze 24/48
            elif 'deberta-v2-xlarge' in model_name:
                print('freeze 12/24')
                self.transformer.embeddings.requires_grad_(False)
                self.transformer.encoder.layer[:12].requires_grad_(False) # freeze 12/24
            elif 'deberta-xlarge' in model_name:
                print('freeze 12/24')
                self.transformer.embeddings.requires_grad_(False)
                self.transformer.encoder.layer[:12].requires_grad_(False) # freeze 12/24
            elif 'funnel-transformer-xlarge' in model_name:
                self.transformer.embeddings.requires_grad_(False)
                self.transformer.encoder.blocks[:1].requires_grad_(False) # freeze 1/3
        
        
        if rnn=='none':
            self.rnn = nn.Identity()
        elif rnn=='lstm':
            self.rnn = LSTMBlock(self.config.hidden_size*self.multi_layers, (self.config.hidden_size*self.multi_layers)//2, num_layers=1, p_drop=p_drop)
        elif rnn=='gru':
            self.rnn = GRUBlock(self.config.hidden_size*self.multi_layers, (self.config.hidden_size*self.multi_layers)//2, num_layers=1, p_drop=p_drop)
        elif rnn=='transformer':
            self.rnn = TransformerBlock(self.config.hidden_size*self.multi_layers, num_layers=1, nhead=8)
        else:
            raise Exception()
    
        if self.msd=='true':
            self.dropout_1 = nn.Dropout(0.1)
            self.dropout_2 = nn.Dropout(0.2)
            self.dropout_3 = nn.Dropout(0.3)
            self.dropout_4 = nn.Dropout(0.4)
            self.dropout_5 = nn.Dropout(0.5)
            
        if head=='simple':
            self.head_types = nn.Sequential(
                nn.Dropout(p_drop),
                nn.Linear(self.config.hidden_size*self.multi_layers, self.num_labels )
            )
        elif head=='norm':
            self.head_types = nn.Sequential(
                nn.LayerNorm(self.config.hidden_size*self.multi_layers),
                nn.Linear(self.config.hidden_size*self.multi_layers, 512),
                nn.GELU(),
                nn.LayerNorm(512),
                nn.Linear(512, self.num_labels ),
            )
        else:
            raise Exception()
        self._init_weights(self.head_types)
        
        
        # ### for multi-task
        # if self.mt=='true':
        self.head_eff = nn.Sequential(
                nn.Dropout(p_drop),
                nn.Linear(self.config.hidden_size*self.multi_layers, self.num_labels_2)
            )
        self._init_weights(self.head_eff)

        
        if loss=='mse':
            self.loss_fn = nn.MSELoss(reduction='none')
        elif loss=='l1':
            self.loss_fn = nn.L1Loss(reduction='none')
        elif loss=='smoothl1':
            self.loss_fn = nn.SmoothL1Loss(reduction='none')
        elif loss=='bce':
            self.loss_fn = nn.BCEWithLogitsLoss(reduction='none')
        elif loss=='focal':
            self.loss_fn = FocalLoss(alpha=1, gamma=0.5, reduction='none')
        elif loss=='xentropy':
            self.loss_fn = nn.CrossEntropyLoss(reduction='none')
        elif loss=='smoothl1+xentropy':
            self.loss_fn = CustomLoss('smoothl1','xentropy',reduction='none')
        else:
            raise Exception()
            
        
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            module.weight.data.normal_(mean=0.0, std=self.config.initializer_range)
            if module.bias is not None:
                module.bias.data.zero_()
        elif isinstance(module, nn.Embedding):
            module.weight.data.normal_(mean=0.0, std=self.config.initializer_range)
            if module.padding_idx is not None:
                module.weight.data[module.padding_idx].zero_()
        elif isinstance(module, nn.LayerNorm):
            module.bias.data.zero_()
            module.weight.data.fill_(1.0)
        

    def forward_logits(self, batch):
        batch_size = batch['input_ids'].size(0)
        assert batch_size == 1, f'Only batch_size=1 supported, got batch_size={batch_size}.'
        
        pooled_out = self.transformer(
            input_ids=batch['input_ids'], attention_mask=batch['attention_mask']).last_hidden_state
        
        # discourse types 
        logits_types = self.head_types(pooled_out)
         
        logits_types_word = aggregate_tokens_to_words(logits_types, batch['word_boxes'])
        assert logits_types_word.size(0) == batch['text'].split().__len__()

        # effectiveness
        logits_eff = self.head_eff(pooled_out) 
        
        logits_eff_word = aggregate_tokens_to_words(logits_eff, batch['word_boxes'])
        assert logits_eff_word.size(0) == batch['text'].split().__len__()

        obj_pred = logits_types_word[..., 0]
        reg_pred = logits_types_word[..., 1:3]
        cls_pred = logits_types_word[..., 3:]

        return obj_pred, reg_pred, cls_pred, logits_types, logits_types_word, logits_eff, logits_eff_word




    def forward_logits_effective(self, input_ids, attention_mask, span_list, aug=False):
        bs = input_ids.size(0)
        assert bs==1, "Batch size is > 1, only bs=1 supported.."
        assert self.multi_layers==1
        
        # sliding window approach to deal with longer tokens than max_length
        # https://www.kaggle.com/competitions/feedback-prize-2021/discussion/313235
        L = input_ids.size(1)
        if self.window_size==-100 or L<=self.window_size:
            x = self.transformer(input_ids=input_ids,
                                 attention_mask=attention_mask).last_hidden_state
        else:
            segments = (L - self.window_size) // self.inner_len
            if (L - self.window_size) % self.inner_len > self.edge_len:
                segments += 1
            elif segments == 0:
                segments += 1
            x = self.transformer(input_ids=input_ids[:,:self.window_size],
                                 attention_mask=attention_mask[:,:self.window_size]).last_hidden_state
            for i in range(1,segments+1):
                start = self.window_size - self.edge_len + (i-1)*self.inner_len
                end   = self.window_size - self.edge_len + (i-1)*self.inner_len + self.window_size
                end = min(end, L)
                x_next = self.transformer(input_ids=input_ids[:,start:end],
                                          attention_mask=attention_mask[:,start:end]).last_hidden_state
                if i==segments:
                    x_next = x_next[:,self.edge_len:]
                else:
                    x_next = x_next[:,self.edge_len:self.edge_len+self.inner_len]
                x = torch.cat([x,x_next], dim=1)
                
        #hidden_states = self.rnn(x) 
        hidden_states = x
        hidden_states = hidden_states.squeeze(0) # (num_tokens,hidden_size*multi_layers)
            
        span_list = span_list[0]
        span_list_next = span_list[1:]+[-1]

        logits_list = []
        for i_token, i_token_next in zip(span_list, span_list_next):
            tmp_logits = hidden_states[i_token:i_token_next,:].mean(dim=0) # (hidden_size*multi_layers)
            logits_list.append(tmp_logits)
        logits = torch.stack(logits_list) # (bs=num_discourse,hidden_size*multi_layers)
        
        # apply rnn
        logits = logits.unsqueeze(0) # (bs=1,num_discourse,hidden_size*multi_layers)
        logits = self.rnn(logits)
        logits = logits.squeeze(0) # (bs=num_discourse,hidden_size*multi_layers)
        
        if aug:
            lam = np.random.beta(self.mixup_alpha, self.mixup_alpha)
            batch_size = logits.size()[0]
            index = torch.randperm(batch_size).cuda()
            logits = lam * logits + (1 - lam) * logits[index, :]
        
        if self.msd=='true' and self.training:
            logits1_1 = self.head_types(self.dropout_1(logits))
            logits1_2 = self.head_types(self.dropout_2(logits))
            logits1_3 = self.head_types(self.dropout_3(logits))
            logits1_4 = self.head_types(self.dropout_4(logits))
            logits1_5 = self.head_types(self.dropout_5(logits))
            logits1 = (logits1_1 + logits1_2 + logits1_3 + logits1_4 + logits1_5) / 5.0
        else:
            logits1 = self.head_types(logits) # (bs,num_labels)
            
        if self.mt=='true': # and self.training:
            if self.msd=='true':
                logits2_1 = self.head_eff(self.dropout_1(logits))
                logits2_2 = self.head_eff(self.dropout_2(logits))
                logits2_3 = self.head_eff(self.dropout_3(logits))
                logits2_4 = self.head_eff(self.dropout_4(logits))
                logits2_5 = self.head_eff(self.dropout_5(logits))
                logits2 = (logits2_1 + logits2_2 + logits2_3 + logits2_4 + logits2_5) / 5.0
            else:
                logits2 = self.head_eff(logits) # (bs,num_labels)
        
        if self.l2norm=='true':
            logits1 = self.s * F.normalize(logits1, dim=-1)
            
        if self.mt=='true': # and self.training:
            return logits1, logits2
        elif aug and self.training:
            return logits1, index, lam
        else:
            return logits1
    
    
    @torch.no_grad()
    def build_target(self, gt_spans, obj_pred, reg_pred, cls_pred, cls_pred_eff):
        obj_target = torch.zeros_like(obj_pred)
        reg_target = torch.zeros_like(reg_pred)
        cls_target = torch.zeros_like(cls_pred)
        cls_eff_target = torch.zeros_like(cls_pred_eff)
        
        # first token as positive
        pos_loc = gt_spans[:, 0]

        obj_target[pos_loc] = 1
        # reg_target[pos_loc, 0] = gt_spans[:, 0].half() if self.cfg.fp16 else gt_spans[:, 0].float()
        # reg_target[pos_loc, 1] = gt_spans[:, 1].half() if self.cfg.fp16 else gt_spans[:, 0].float()
        reg_target[pos_loc, 0] = gt_spans[:, 0].float()
        reg_target[pos_loc, 1] = gt_spans[:, 0].float()
        cls_target[pos_loc, gt_spans[:, 2]] = 1
        cls_eff_target[pos_loc, gt_spans[:, 3]] = 1
        
        # ### ++
        # if self.num_classes2==1:
        #     cls_eff_target[pos_loc, 0] = gt_spans[:, 3].half() if self.cfg.fp16 else gt_spans[:, 3].float()
        # else:
        #     cls_eff_target[pos_loc, gt_spans[:, 3]] = 1

        # dynamically assign one more positive
        if self.dynamic_positive:
            
            cls_prob = (obj_pred.sigmoid().unsqueeze(1) * cls_pred.sigmoid()).sqrt()
            # cls_prob = (obj_pred.unsqueeze(1) * cls_pred).sqrt() ## for BCEwithlogits

            for start, end, label, label_eff in gt_spans:
                _cls_prob = cls_prob[start:end]
                _cls_gt = _cls_prob.new_full((_cls_prob.size(0), ),
                                             label,
                                             dtype=torch.long)
                _cls_gt = F.one_hot(
                    _cls_gt, num_classes=_cls_prob.size(1)).type_as(_cls_prob)
                
                cls_cost = F.binary_cross_entropy(_cls_prob, _cls_gt, reduction='none').sum(-1)
                ## Try with BCELogits ?
                # cls_cost = F.binary_cross_entropy_with_logits(_cls_prob, _cls_gt, reduction='none').sum(-1)

                ### TODO add cls_cost_effective here!! 
                ### ... 


                _reg_pred = reg_pred[start:end].exp()
                _reg_loc = torch.arange(_reg_pred.size(0),
                                        device=_reg_pred.device)
                px1 = _reg_loc - _reg_pred[:, 0]
                px2 = _reg_loc + _reg_pred[:, 1]
                ix1 = torch.max(px1, _reg_loc[0])
                ix2 = torch.min(px2, _reg_loc[-1])
                ux1 = torch.min(px1, _reg_loc[0])
                ux2 = torch.max(px2, _reg_loc[-1])
                inter = (ix2 - ix1).clamp(min=0)
                union = (ux2 - ux1).clamp(min=0) + 1e-12
                iou = inter / union
                iou_cost = -torch.log(iou + 1e-12)
                cost = cls_cost + iou_cost

                pos_ind = start + cost.argmin()
                obj_target[pos_ind] = 1
                reg_target[pos_ind, 0] = start
                reg_target[pos_ind, 1] = end
                cls_target[pos_ind, label] = 1
            
                ## ++ TODO try next run 
                # cls_eff_target[pos_ind, label_eff] = 1
            
            pos_loc = (obj_target == 1).nonzero().flatten()
        
        return obj_target, reg_target, cls_target, cls_eff_target, pos_loc # 



  


    def training_step(self, batch):
        if self.device=="cuda": 
            batch = to_gpu(batch)

        input_data = {
            'input_ids': batch['input_ids'],
            'attention_mask': batch['attention_mask'],
            'span_list': batch['span_list'],
            'aug':False,
        }
        
        # get loss
        if self.loss in ['xentropy']:
            if self.mt=='true': # and self.training:
                # logits, logits2 = self.forward_logits_effective(**input_data)
                obj_pred, reg_pred, cls_pred, logits_types, logits_types_word, logits_eff, logits_eff_word = self.forward_logits(batch)

                obj_target, reg_target, cls_target, cls_eff_target, pos_loc = self.build_target(
                    batch['gt_spans'], obj_pred, reg_pred, cls_pred, logits_eff_word)

                obj_loss, reg_loss, cls_loss, cls_eff_loss = self.get_losses(
                    obj_pred, reg_pred, cls_pred, logits_eff_word,
                    obj_target, reg_target, cls_target, cls_eff_target, pos_loc
                    )
                
                loss1 = obj_loss + reg_loss + cls_loss
                loss2 = cls_eff_loss
                # loss = self.get_losses(logits, data['label']).mean()
                # loss += self.w_mt * self.get_losses2(logits2, data['label2']).mean()
                loss = loss1 + self.w_mt * cls_eff_loss
            
            elif self.aug=='mixup' and np.random.random()<self.p_aug and self._current_epoch<self.aug_stop_epoch and self.training:
                input_data['aug'] = True
                logits, index, lam = self.forward_logits(**input_data)
                loss_a = self.get_losses(logits, batch['label']).mean()
                loss_b = self.get_losses(logits, batch['label'][index]).mean()
                loss = lam * loss_a + (1 - lam) * loss_b
            else:
                logits = self.forward_logits(**input_data)
                loss = self.get_losses(logits, batch['label']).mean()
        else:
            raise Exception()

        ## TODO    
        if self.adv_sift=='true':
            input_data['aug'] = False
            loss = loss + self.adv.loss(logits, self.logits_fn, **input_data)
        
        # # get Eff score
        # pred = logits.softmax(dim=-1).detach().cpu().numpy() # (bs=num_discourse,hidden_size*multi_layers)
        # label = data['label'].detach().cpu().numpy()

        return {
            'loss':loss,
            'loss1':loss1,
            'loss2':loss2,
            'obj_loss':obj_loss.item(),
            'reg_loss':reg_loss.item(),
            'cls_loss':cls_loss.item(),
            # 'score':score,
            # 'pred':pred,
            # 'label':label,
            # ####
            # 'logits_types': logits_types,
            # 'logits_types_word': logits_types_word,
            # 'logits_eff': logits_eff,
            # 'logits_eff_word': logits_eff_word,
        }
        # return pred, label, loss
    



    
    def validation_step(self, batch):
        if self.device=="cuda":  batch = to_gpu(batch)
        input_data = {
            'input_ids': batch['input_ids'],
            'attention_mask': batch['attention_mask'],
            'span_list': batch['span_list'],
            'aug':False,
        }
        
        # get loss
        # if self.loss in ['xentropy']:
            # if self.mt=='true': # and self.training:
        obj_pred, reg_pred, cls_pred, logits_types, logits_types_word, logits_eff, logits_eff_word = self.forward_logits(batch)

        obj_target, reg_target, cls_target, cls_eff_target, pos_loc = self.build_target(
            batch['gt_spans'], obj_pred, reg_pred, cls_pred, logits_eff_word)

        obj_loss, reg_loss, cls_loss, cls_eff_loss = self.get_losses(
            obj_pred, reg_pred, cls_pred, logits_eff_word,
            obj_target, reg_target, cls_target, cls_eff_target, pos_loc
            )
        
        loss1 = obj_loss + reg_loss + cls_loss
        loss2 = cls_eff_loss
        # loss = self.get_losses(logits, data['label']).mean()
        # loss += self.w_mt * self.get_losses2(logits2, data['label2']).mean()
        loss = loss1 + self.w_mt * cls_eff_loss

        self.val_outputs.append({
            'loss':loss,
            'loss1':loss1,
            'loss2':loss2,
            'obj_loss':obj_loss.item(),
            'reg_loss':reg_loss.item(),
            'cls_loss':cls_loss.item(),
            # 'score':score,
            # 'pred':pred,
            # 'label':label,
            # ####
            # 'logits_types': logits_types,
            # 'logits_types_word': logits_types_word,
            # 'logits_eff': logits_eff,
            # 'logits_eff_word': logits_eff_word,
            'text':batch['text'],
            'text_id':batch['text_id'],
        })

        # get pred effectiveness
        # pred = logits.softmax(dim=-1).detach().cpu().numpy().reshape(-1, self.num_labels)

        res = self.predict_step(batch, self.test_score_thr)
        gt_df = valid_df[valid_df['id']==res['text_id']].reset_index(drop=True)
        if len(res.keys())>1:
            # pred = []
            # text_id = res['text_id']
            # for c,start,end in zip(res['label'],res['start'],res['end']):
            #     pred.append([text_id, LABEL2TYPE[c], ' '.join(np.arange(start,end+1).astype(str))])
            # pred_df = pd.DataFrame(pred, columns=['id','class','predictionstring'])
            pred_df = format_result(res)
        else:
            # pred_df = pd.DataFrame([], columns=['id','class','predictionstring'])
            pred_df = pd.DataFrame([], columns=[
                'id','class','predictionstring', 'score', 'class', 'class_eff',
                'score_eff', 'score_eff_0', 'score_eff_1'])

        return {
            'loss': loss,
            'loss1': loss1,
            'loss2': loss2,
            'obj_loss': obj_loss.item(),
            'reg_loss': reg_loss.item(),
            'cls_loss': cls_loss.item(),
            'text': batch['text'],
            'text_id': batch['text_id'],
            'pred_df': pred_df,
            'gt_df': gt_df,
            # 'pred':pred,
            # 'label': batch['label'].detach().cpu().numpy().reshape(-1,),
        }
    
    
    def validation_epoch_end(self, outputs):
        # losses = []
        # preds = []
        # labels = []
        # for o in outputs:
        #     losses.append(o['loss'])
        #     preds.append(o['pred'])
        #     labels.append(o['label'])
        # losses = np.hstack(losses).mean()
        # preds = np.vstack(preds)
        # labels = np.hstack(labels)
        # scores = self.get_scores(preds, labels)

        losses = []
        obj_losses = []
        reg_losses = []
        cls_losses = []
        pred_dfs = []
        gt_dfs = []
        for o in outputs:
            losses.append(o['loss'])
            obj_losses.append(o['obj_loss'])
            reg_losses.append(o['reg_loss'])
            cls_losses.append(o['cls_loss'])
            pred_dfs.append(o['pred_df'])
            gt_dfs.append(o['gt_df'])
        loss = np.array(losses).mean()
        obj_loss = np.array(obj_losses).mean()
        reg_loss = np.array(reg_losses).mean()
        cls_loss = np.array(cls_losses).mean()
        
        pred_df = pd.concat(pred_dfs, axis=0).reset_index(drop=True)
        gt_df = pd.concat(gt_dfs, axis=0).reset_index(drop=True)
        # avg_f1 = score_feedback_comp(pred_df, gt_df, return_class_scores=False)
        avg_f1 = get_f1_score(pred_df, gt_df, True)
        
        if avg_f1 > self.best_score:
            self.best_score = avg_f1     
        ### reset placeholder 
        self.val_outputs = []
        self._current_epoch += 1

        # return losses, scores
        return {
            'loss': loss,
            'obj_loss': obj_loss,
            'reg_loss': reg_loss,
            'cls_loss': cls_loss,
            'f1': avg_f1,
            'best_f1': self.best_score,
        }



    def predict_step(self, batch, test_score_thr):
        if self.device == "cuda": batch = to_gpu(batch)

        obj_pred, reg_pred, cls_pred, _, _, _, logits_eff_word = self.forward_logits(batch)
        obj_pred = obj_pred.sigmoid()
        reg_pred = reg_pred.exp()
        cls_pred = cls_pred.sigmoid()

        obj_scores = obj_pred
        cls_scores, cls_labels = cls_pred.max(-1)
        pr_scores = (obj_scores * cls_scores)**0.5
        pos_inds = pr_scores > test_score_thr

        if pos_inds.sum() == 0:
            return dict(text_id=batch['text_id'])

        pr_score, pr_label = pr_scores[pos_inds], cls_labels[pos_inds]
        pos_loc = pos_inds.nonzero().flatten()
        start = pos_loc - reg_pred[pos_inds, 0]
        end = pos_loc + reg_pred[pos_inds, 1]

        min_idx, max_idx = 0, obj_pred.numel() - 1
        start = start.clamp(min=min_idx, max=max_idx).round().long()
        end = end.clamp(min=min_idx, max=max_idx).round().long()

        # nms
        keep = span_nms(start, end, pr_score)
        start = start[keep]
        end = end[keep]
        pr_score = pr_score[keep]
        pr_label = pr_label[keep]


        ### incl effectiveness preds ## TODO try with 
        if self.use_sigmoid:
            logits_eff_word = logits_eff_word.sigmoid()         ## run 1 (2 classes)
            eff_proba, eff_cls = logits_eff_word.max(-1) 

            eff_proba = eff_proba[keep]
            eff_cls   = eff_cls[keep]

            # eff_proba_0 = eff_proba_1 = eff_proba
            eff_proba_0 = logits_eff_word[keep, 0] ## negative class
            eff_proba_1 = logits_eff_word[keep, 1] ## positive class

        else:
            ### use softmax 
            logits_eff_word = torch.softmax(logits_eff_word, -1)  ## run 2 (2 classes)

            _, eff_cls = logits_eff_word.max(-1) 
            eff_proba_1 = logits_eff_word[keep, 1] ## positive class
            eff_proba_0 = logits_eff_word[keep, 0] ## negative class
            # eff_proba = eff_proba[keep]
            eff_proba = eff_proba_1

            eff_cls   = eff_cls[keep]

        ###########################################################
        # # eff_proba, eff_cls = logits_eff_word[start:end].max(-1) 
        # eff_proba = []
        # eff_cls = []
        # for st_, en_ in zip(start, end):
        #     ## apply first mean then max
        #     active_logits_eff_word = logits_eff_word[st_:en_].mean()
        #     eff_proba_, eff_cls_ = active_logits_eff_word.max(-1) 

        #     eff_proba.append(eff_proba_.detach().cpu().numpy())
        #     eff_cls.append(eff_cls_.detach().cpu().numpy())


        return dict(
            text_id=batch['text_id'],
            start=to_np(start),
            end=to_np(end),
            score=to_np(pr_score),
            label=to_np(pr_label),
            ###
            label_eff=to_np(eff_cls),
            score_eff=to_np(eff_proba),
            ###
            score_eff_0=to_np(eff_proba_0),
            score_eff_1=to_np(eff_proba_1),
            )

    
    def test_step_effective(self, batch):
        data = to_gpu(batch)
        input_data = {
            'input_ids':data['input_ids'],
            'attention_mask':data['attention_mask'],
            'span_list':data['span_list'],
            'aug':False,
        }
        logits = self.forward_logits_effective(**input_data)
        pred = logits.softmax(dim=-1).detach().cpu().numpy().reshape(-1,self.num_labels)
        return {
            'pred':pred,
            'discourse_ids':data['discourse_ids'],
            'text':data['text'],
            'essay_id':data['essay_id'],
        }
        
    def configure_optimizers(self):
        optimizer = self.fetch_optimizer()
        scheduler = self.fetch_scheduler(optimizer)
        scheduler = {"scheduler": scheduler, "interval": "step", "frequency": 1}
        return [optimizer], [scheduler]
    
    def fetch_optimizer(self):
        head_params = list(self.head_types.named_parameters()) + list(self.rnn.named_parameters())
        head_params += list(self.head_eff.named_parameters())

        param_optimizer = list(self.transformer.named_parameters())
        no_decay = ["bias", "LayerNorm.bias"]
        optimizer_parameters = [
            {
                "params": [p for n, p in param_optimizer if not any(nd in n for nd in no_decay)],
                "weight_decay": 0.01,
            },
            {
                "params": [p for n, p in param_optimizer if any(nd in n for nd in no_decay)],
                "weight_decay": 0.0,
            },
            {
                "params": [p for n,p in head_params], 
                "weight_decay": 0.01,
                "lr": self.head_learning_rate,
            },
        ]
        # if self.mt=='true':
        #     head2_params = list(self.head_eff.named_parameters())
        #     optimizer_parameters.append(
        #         {
        #             "params": [p for n,p in head2_params], 
        #             "weight_decay": 0.01,
        #             "lr": self.head_learning_rate,
        #         } )
        
        optimizer = AdamW(optimizer_parameters, lr=self.learning_rate)
        
        # # https://www.kaggle.com/code/nbroad/8-bit-adam-optimization/notebook
        # # These are the only changes you need to make
        # # The first part sets the optimizer to use 8-bits
        # # The for loop sets embeddings to use 32-bits
        # if self.adam_bits == 32:
        #     optimizer = bnb.optim.AdamW32bit(optimizer_parameters, lr=self.learning_rate)
        # if self.adam_bits == 8:
        #     optimizer = bnb.optim.AdamW8bit(optimizer_parameters, lr=self.learning_rate)
        #     # Thank you @gregorlied https://www.kaggle.com/nbroad/8-bit-adam-optimization/comments#1661976
        #     for module in self.transformer.modules():
        #         if isinstance(module, torch.nn.Embedding):
        #             bnb.optim.GlobalOptimManager.get_instance().register_module_override(
        #                 module, 'weight', {'optim_bits': 32}
        #             )   
        
        return optimizer

    def fetch_scheduler(self, optimizer):
        print('self.warmup_ratio = ', self.warmup_ratio)
        print('self.num_train_steps = ', self.num_train_steps)
        print('num_warmup_steps = ', int(self.warmup_ratio * self.num_train_steps))
        if self.scheduler=='cosine':
            scheduler = get_cosine_schedule_with_warmup(
                optimizer,
                num_warmup_steps=int(self.warmup_ratio * self.num_train_steps),
                num_training_steps=self.num_train_steps,
                num_cycles=0.5*self.num_cycles,
                last_epoch=-1,
            )
        elif self.scheduler=='cosine_hard':
            scheduler = get_cosine_with_hard_restarts_schedule_with_warmup(
                optimizer,
                num_warmup_steps=int(self.warmup_ratio * self.num_train_steps),
                num_training_steps=self.num_train_steps,
                num_cycles=self.num_cycles,
                last_epoch=-1,
            )
        elif self.scheduler=='linear':
            scheduler = get_linear_schedule_with_warmup(
                optimizer,
                num_warmup_steps=int(self.warmup_ratio * self.num_train_steps),
                num_training_steps=self.num_train_steps,
                last_epoch=-1,
            )
        return scheduler
    

    def get_ce_loss(self, logits, label):
        if self.loss in ['xentropy']:
            loss = self.loss_fn(logits.reshape(-1,self.num_labels), label.reshape(-1,))
        else:
            raise Exception()
        return loss

    
    
    def get_scores_eff(self, pred, label):
        score = -1 * log_loss(label.reshape(-1,), pred.reshape(-1,self.num_labels))
        return score
    
    def logits_fn(self, *wargs, **kwargs):
        if self.mt=='true':
            logits,_ = self.forward_logits(**kwargs)
        else:
            logits = self.forward_logits(**kwargs)
        return logits


    def get_losses(self, obj_pred, reg_pred, cls_pred, logits_eff_word, 
                obj_target, reg_target, cls_target, cls_eff_target, pos_loc):
            
            num_total_samples = pos_loc.numel()
            assert num_total_samples > 0
            reg_pred = reg_pred[pos_loc].exp()
            reg_target = reg_target[pos_loc]
            px1 = pos_loc - reg_pred[:, 0]
            px2 = pos_loc + reg_pred[:, 1]
            gx1 = reg_target[:, 0]
            gx2 = reg_target[:, 1]
            ix1 = torch.max(px1, gx1)
            ix2 = torch.min(px2, gx2)
            ux1 = torch.min(px1, gx1)
            ux2 = torch.max(px2, gx2)
            inter = (ix2 - ix1).clamp(min=0)
            union = (ux2 - ux1).clamp(min=0) + 1e-12
            iou = inter / union
                        
            # reg_loss = -iou.log().sum() / num_total_samples
            ### TODO fix 
            ioulog = -iou.log()  
            ioulog[ioulog == float('inf')] = 0 # replace inf values with zero in iou tensor
            reg_loss = ioulog.sum() / num_total_samples
            
            cls_loss = F.binary_cross_entropy_with_logits(
                cls_pred[pos_loc],
                cls_target[pos_loc] * iou.detach().reshape(-1, 1),
                reduction='sum') / num_total_samples
            
            obj_loss = F.binary_cross_entropy_with_logits(
                obj_pred, obj_target, reduction='sum') / num_total_samples
            
            # cls_loss = cls_loss_bce

            ##### ++ Jaccard loss 
            # loss_fct_j = JaccardLoss()
            # cls_loss_j = loss_fct_j(cls_pred[pos_loc], cls_target[pos_loc] * iou.detach().reshape(-1, 1)) / num_total_samples

            # ### FOCAL Loss
            # gamma = 2.0
            # alpha = 1.0
            # loss_fct1 = nn.BCEWithLogitsLoss(reduction='none')
            # cls_bce_l1 = loss_fct1(cls_pred[pos_loc], cls_target[pos_loc] * iou.detach().reshape(-1, 1))
            # probas = torch.sigmoid(cls_pred[pos_loc])
            # cls_loss_j = torch.where(cls_target[pos_loc] >= 0.5, alpha * (1. - probas)**gamma * cls_bce_l1, probas**gamma * cls_bce_l1)
            # cls_loss_j = cls_loss_j.mean()
            # cls_loss = cls_loss_bce + cls_loss_j

            # ###### ++
            # cls_eff_loss = F.binary_cross_entropy_with_logits(
            #     logits_eff_word[pos_loc], cls_eff_target[pos_loc],
            #     #cls_eff_target[pos_loc] * iou.detach().reshape(-1, 1), ## TODO check 
            #     reduction='sum') / num_total_samples

            loss_fct = nn.CrossEntropyLoss(reduction='sum')
            cls_eff_loss_ce = loss_fct(
                logits_eff_word[pos_loc].softmax(-1), 
                cls_eff_target[pos_loc]
                # cls_eff_target[pos_loc] * iou.detach().reshape(-1, 1) # TODO try!!
                ) / num_total_samples
            
            # cls_eff_loss = F.cross_entropy(logits_eff_word[pos_loc].softmax(-1), cls_eff_target[pos_loc], reduction='sum') / num_total_samples

            # loss_fct_bce = nn.BCEWithLogitsLoss(reduction='sum', pos_weight=torch.tensor([3.0]).cuda())
            # cls_eff_loss_bce = loss_fct_bce(
            #     logits_eff_word[pos_loc], 
            #     cls_eff_target[pos_loc] 
            #     ) / num_total_samples
            
            cls_eff_loss = cls_eff_loss_ce


            # loss_fct2 = JaccardLoss()
            # cls_eff_loss_j = loss_fct2(logits_eff_word[pos_loc], cls_eff_target[pos_loc]) / num_total_samples
            
            # ### FOCAL Loss 
            # gamma = 2.0 
            # alpha = 1.0 
            # loss_fct2 = nn.BCEWithLogitsLoss(reduction='none')
            # bce_l1 = loss_fct2(logits_eff_word[pos_loc], cls_eff_target[pos_loc])
            # probas = torch.sigmoid(logits_eff_word[pos_loc])
            # cls_eff_loss_focal = torch.where(cls_eff_target[pos_loc] >= 0.5, alpha * (1. - probas)**gamma * bce_l1, probas**gamma * bce_l1)
            # cls_eff_loss_focal = cls_eff_loss_focal.mean()
            
            # cls_eff_loss = cls_eff_loss_bce + cls_eff_loss_focal

            return obj_loss, reg_loss, cls_loss, cls_eff_loss



In [ ]:
# model_pretraining = None     
# if 'tascj' in CFG.pretrained_detector_path:
#     # from models_detector import TextSpanDetectorOriginal
#     model_pretraining = TextSpanDetectorOriginal(CFG.model_name, num_classes=7)
#     model_pretraining.load_state_dict(torch.load(CFG.pretrained_detector_path))

# elif CFG.pretrained_detector_path != 'none':
#     # from models_detector import TextSpanDetector
#     model_pretraining = TextSpanDetector(
#         CFG.model_name,
#         tokenizer,
#         num_classes=7,
#         dynamic_positive=True,
#         with_cp=False,
#         rnn=RNN,
#         loss=LOSS,
#         head=HEAD,
#         multi_layers=1,
#     )
#     model_pretraining.load_state_dict(torch.load(CFG.pretrained_detector_path))

# model = WoSpanDetector(CFG.model_name, 
#                 tokenizer,
#                 num_labels=7, #3 
#                 num_labels_2=2,
#                 mt='true')
# model = model.to(device)
# # if CFG.pretrain_path != 'none':
# #     model.load_state_dict(torch.load(CFG.pretrain_path))

# Infer Test

In [ ]:
# CFG.infer = True
CFG.infer = False

In [ ]:
class SpanDetectorDatasetTest(Dataset):
    def __init__(self, df, tokenizer,):
        
        self.df = df.copy() #pd.read_csv(csv_file)
        self.samples = list(self.df.groupby('id'))
        self.tokenizer = tokenizer
        print(f'Loaded {len(self)} samples.')


    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        text_id, df = self.samples[index]
        
        # text_path = os.path.join(self.text_dir, f'{text_id}.txt')
        # with open(text_path) as f:
        #     text = f.read().rstrip()
        text = df['essay_text'].values[0].rstrip()

        tokens = self.tokenizer(text, return_offsets_mapping=True)
        
        input_ids = torch.LongTensor(tokens['input_ids'])
        attention_mask = torch.LongTensor(tokens['attention_mask'])
        offset_mapping = np.array(tokens['offset_mapping'])
        offset_mapping = self.strip_offset_mapping(text, offset_mapping)
        num_tokens = len(input_ids)

        # token slices of words
        woff = self.get_word_offsets(text)
        toff = offset_mapping
        wx1, wx2 = woff.T
        tx1, tx2 = toff.T
        ix1 = np.maximum(wx1[..., None], tx1[None, ...])
        ix2 = np.minimum(wx2[..., None], tx2[None, ...])
        ux1 = np.minimum(wx1[..., None], tx1[None, ...])
        ux2 = np.maximum(wx2[..., None], tx2[None, ...])
        ious = (ix2 - ix1).clip(min=0) / (ux2 - ux1 + 1e-12)
        assert (ious > 0).any(-1).all()

        word_boxes = []
        for row in ious:
            inds = row.nonzero()[0]
            word_boxes.append([inds[0], 0, inds[-1] + 1, 1])
        word_boxes = torch.FloatTensor(word_boxes)


        return dict(
            text=text,
            text_id=text_id,
            input_ids=input_ids,
            attention_mask=attention_mask,
            word_boxes=word_boxes)
    

    def strip_offset_mapping(self, text, offset_mapping):
        ret = []
        for start, end in offset_mapping:
            match = list(re.finditer('\S+', text[start:end]))
            if len(match) == 0:
                ret.append((start, end))
            else:
                span_start, span_end = match[0].span()
                ret.append((start + span_start, start + span_end))
        return np.array(ret)

    def get_word_offsets(self, text):
        matches = re.finditer("\S+", text)
        spans = []
        words = []
        for match in matches:
            span = match.span()
            word = match.group()
            spans.append(span)
            words.append(word)
        assert tuple(words) == tuple(text.split())
        return np.array(spans)
    


class CustomCollatorTest(object):
    def __init__(self, tokenizer, model_name): #model
        self.pad_token_id = tokenizer.pad_token_id

        config_model = config = AutoConfig.from_pretrained(model_name)
        # if hasattr(model.config, 'attention_window'):
        if hasattr(config_model, 'attention_window'):
            # For longformer
            # https://github.com/huggingface/transformers/blob/v4.17.0/src/transformers/models/longformer/modeling_longformer.py#L1548
            self.attention_window = (
                config_model.attention_window if isinstance(
                    config_model.attention_window, int)
                else max(config_model.attention_window)
                )
        else:
            self.attention_window = None

    def __call__(self, samples):
        batch_size = len(samples)
        assert batch_size == 1, f'Only batch_size=1 supported, got batch_size={batch_size}.'

        sample = samples[0]

        max_seq_length = len(sample['input_ids'])
        if self.attention_window is not None:
            attention_window = self.attention_window
            padded_length = (attention_window -
                             max_seq_length % attention_window
                             ) % attention_window + max_seq_length
        else:
            padded_length = max_seq_length

        input_shape = (1, padded_length)
        input_ids = torch.full(input_shape,
                               self.pad_token_id,
                               dtype=torch.long)
        attention_mask = torch.zeros(input_shape, dtype=torch.long)
 
        seq_length = len(sample['input_ids'])
        input_ids[0, :seq_length] = sample['input_ids']
        attention_mask[0, :seq_length] = sample['attention_mask']
 
        text_id = sample['text_id']
        text = sample['text']
        word_boxes = sample['word_boxes']
        # gt_spans = sample['gt_spans']

        return dict(
            text_id=text_id,
            text=text,
            input_ids=input_ids,
            attention_mask=attention_mask,
            word_boxes=word_boxes,
            )
    

In [ ]:
def generate_submission(df_sub, save_name=None, out_dir=OUTPUT, verbose=False):

    pred_df = df_sub.copy()
    print(pred_df.shape)
    print(pred_df.id.nunique())
    display(pred_df)

    # ## convert effective scores to RATER format 
    # pred_df['score_discourse_effectiveness_0'] = -100
    # pred_df['score_discourse_effectiveness_1'] = -100
    # for i, row in pred_df.iterrows():
    #     if row['class_eff'] == 2: #'Effective':  
    #         pred_df.loc[i, 'score_discourse_effectiveness_1'] = row['score_eff']
    #         pred_df.loc[i, 'score_discourse_effectiveness_0'] = 1 - row['score_eff']
    #     else:
    #         pred_df.loc[i, 'score_discourse_effectiveness_0'] = row['score_eff']
    #         pred_df.loc[i, 'score_discourse_effectiveness_1'] = 1 - row['score_eff']


    sample_sub = pd.read_csv(DIR / 'sample_submission.csv')

    sub = pred_df.copy()
    sub = sub.rename(columns={'id': 'essay_id_comp'})

    sub['score_discourse_effectiveness_0'] = pred_df['score_eff_0']
    sub['score_discourse_effectiveness_1'] = pred_df['score_eff_1']


    # sub["discourse_type"] = sub["class"].map(discourse_type_to_int)
    # sub["discourse_type"] = sub["discourse_type"].map(discourse_type_to_int)
    sub["discourse_type"] = sub["class"]

    sub = sub[sample_sub.columns]
    # if sub.essay_id_comp.nunique() == 10402:
    #     sub.to_csv(f'{out_dir}/{save_name}.csv', index=False)
    # else:
    #     print("check ids", sub.essay_id_comp.nunique())

    display(sub.head())
    print(sub.essay_id_comp.nunique() == 10402)
    
    ###
    if verbose:
        print(pred_df['class'].value_counts())
        print()
        print(pred_df['class_eff'].value_counts())
        print(pred_df['class_eff'].value_counts(normalize=True))
        print(sub.dtypes)

        import seaborn as sns 
        import matplotlib.pyplot as plt

        sns.histplot(sub.score_discourse_effectiveness_0)
        sns.histplot(sub.score_discourse_effectiveness_1)
        plt.show()

    return sub

In [ ]:
if CFG.infer:

    test = pd.read_csv('/mnt/md1/Work/dev_IME/Git_hacks/Feedback3/input/test.csv')
    test.columns = ['id', 'essay_text']
    test['essay_id'] = test['id'] 

    print(test.shape)
    display(test.head())

    test_ds = SpanDetectorDatasetTest(test, tokenizer=CFG.tokenizer)
    # print(test_ds[0].keys())
    
    test_loader = DataLoader(test_ds,
                            batch_size=1,
                            shuffle=False,
                            num_workers=2,
                            collate_fn=CustomCollatorTest(CFG.tokenizer, CFG.model_name)
                            )

In [ ]:
if CFG.infer:

    weight_path = ""

    # print('test th:', model.test_score_thr)
    model = TextSpanDetector(
            args.model,
            tokenizer,
            num_classes=args.num_classes,
            dynamic_positive=True,
            with_cp=(args.check_pointing=='true'),
            hidden_dropout_prob=args.hidden_drop_prob, 
            p_drop=args.p_drop,
            scheduler=args.scheduler,
            learning_rate=args.lr,
            head_learning_rate=args.head_lr,
            # num_train_steps=num_train_steps,
            # warmup_ratio=args.warmup_ratio,
            # model_pretraining=model_pretraining,
        )
    model.load_state_dict(torch.load(weight_path))
    model = model.cuda()
    model.eval()

    print('test th:', model.test_score_thr)
    
    outputs = []
    for batch_idx, batch in tqdm(enumerate(test_loader), total=len(test_loader)):
        with torch.no_grad():
            output = model.test_step(batch)
            outputs.append(output)
    del model
    torch.cuda.empty_cache()
            
    pred_dfs = []
    discourse_ids = []
    prob_seqs = []
    for o in outputs:
        pred_dfs.append(o['pred_df'])
        # discourse_ids.extend(o['discourse_ids'])
        # prob_seqs.extend(o['prob_seq'])
    # preds = np.vstack(preds)
        
    pred_df = pd.concat(pred_dfs, axis=0).reset_index(drop=True)

    # res = trainer.predict(model, test_loader, return_predictions=True)  
    # pred_df = pd.concat([format_result(res[i]) for i in range(len(res)) if res[i] is not None])

    ####
    # sub = generate_submission(pred_df, test)

In [ ]:
pred_df

In [ ]:
pred_df.id.nunique()

In [ ]:
pred_df['class_eff'].value_counts()

In [ ]:
# sns.histplot(data=pred_df, x='score_eff_0', bins=50, kde=True)
# sns.histplot(data=pred_df, x='score_eff_1', bins=50, kde=True)

In [ ]:
# pred_df.rename(columns={
#     "score_eff_0": "score_discourse_effectiveness_0",
#     "score_eff_1": "score_discourse_effectiveness_1",
# }).to_csv(OUTPUT / f"test_preds_fold2_th05.csv", index=False)

In [ ]:
sub = generate_submission(pred_df)

In [ ]:
### Check missing ids 
missing_ids = list(set(test.id) - set(sub.essay_id_comp))
print(missing_ids)

if len(missing_ids)>0:

    dummy_sub = pd.read_csv('/mnt/md1/Work/dev_IME/Git_hacks/Feedback3/submissions/sub_wbf_exp18_exp22_6ckp.csv')
    dummy_sub = dummy_sub[dummy_sub.essay_id_comp.isin(missing_ids)]
    dummy_sub = dummy_sub.reset_index(drop=True)
    display(dummy_sub)

    sub = pd.concat([sub, dummy_sub], ignore_index=True)

    print(sub.essay_id_comp.nunique() == 10402)

    # ['A32121C1963B', '17025BA7A3B3', 'E8C25EDACD20']
    # ['DEC69C9D9F72', '93E547BF9654', 'CCD8783CF958', '261DCD26AE0C', 'E8C25EDACD20', '178BED4E1487', '7572D2FA0521', '7EEF18A458A3']

In [ ]:
import seaborn as sns 

sns.histplot(data=sub, x='score_discourse_effectiveness_0', bins=50, kde=True)
sns.histplot(data=sub, x='score_discourse_effectiveness_1', bins=50, kde=True)

In [ ]:
sub['discourse_type'].value_counts()

In [ ]:
sub.isnull().sum()

In [ ]:
model.test_score_thr

In [ ]:
# sub.to_csv(OUTPUT / f"sub_fold0_th05_ckp0.csv", index=False)
# sub.to_csv(OUTPUT / f"sub_fold0_th05_ckp1.csv", index=False)